# DuckDB extensions · from intent to first result

**Interactive journey design · 27 October 2026**

The brief carries forward our established direction: a Jute notebook sidebar for people using DuckDB, every unique Core and Community extension discoverable, and one continuous path from a task to useful query results.

**Audience:** notebook users who know their task but may not know the extension or function. **Surface:** desktop notebook + extension sidebar, responsive at narrow widths. **Tone:** clear, compact, calm. **Success:** see a useful result, understand which notebook is ready, and recover without losing inputs.

**Design direction:** Modern Minimal, a light software interface using the bundled Open Design palette and system fonts. Variance 3, motion 2, density 6. The primary accent identifies the main action; borders and typography establish hierarchy.

**Deliverable:** a clickable design prototype, complete journey and recovery map, task recipe contracts, source-based findings, and bounded solver evidence. Setup, credentials and results in the prototype are simulated. No extension installation or provider request is performed by the prototype.

**Decisions:** Core / Community identifies origin. Readiness belongs to the active runtime. Recipes are optional guidance and do not gate generic installation. Query execution remains explicit.

**Start here:** open the **Discovery & Choice** output below the original prototype. Search by task, extension or function; filter Core / Community; compare **Existing UI**, **To-be design** or **Side-by-side**; then select **Choose this task · Try journey**. The selected extension opens in the shared journey below the comparison. On narrow screens the comparison stacks vertically. Choices remain in the output session and reset when the cell reruns.

## Build sequence

1. Map discovery, choice, inputs, preparation, execution, return and recovery.
2. Build a working extension browser and task flow inside one HTML output.
3. Add scenarios for stopped kernels, installation failure, missing credentials, incompatible versions, no recipe and runtime restart.
4. Verify bounded transitions with `spurpower-solve` and inspect the rendered prototype.

Enable **active content** on the HTML output to use the controls. The initial view and a journey summary remain readable with scripts disabled.


In [14]:

var duckdbJourneyBuild = await (async () => {
 const {h} = await import("https://esm.sh/preact@10.24.3");
 const {render:ssr} = await import("https://esm.sh/preact-render-to-string@6.5.11");
 const esbuild = await import("npm:esbuild@0.24.0");
 const {JSDOM} = await import("npm:jsdom@24.1.3");
 const viewSource = "\nimport { h } from \"https://esm.sh/preact@10.24.3\";\nexport const entries = [\n {id:\"excel\",name:\"Excel\",source:\"Core\",purpose:\"Read Excel worksheets\",description:\"Bring an .xlsx worksheet into a SQL query.\",tags:\"spreadsheet import xlsx read file worksheet\",task:\"Read a worksheet\",recipe:true},\n {id:\"httpfs\",name:\"HTTPFS\",source:\"Core\",purpose:\"Query files from a URL\",description:\"Read remote Parquet files without a separate import step.\",tags:\"https remote parquet s3 cloud url\",task:\"Preview remote Parquet\",recipe:true},\n {id:\"spatial\",name:\"Spatial\",source:\"Core\",purpose:\"Work with geographic data\",description:\"Create geometries and query spatial relationships.\",tags:\"map location geography point latitude longitude gis\",task:\"Create a geographic point\",recipe:true},\n {id:\"h3\",name:\"H3\",source:\"Community\",purpose:\"Index coordinates into a grid\",description:\"Turn a latitude and longitude into an H3 cell.\",tags:\"hexagon grid map location coordinates latitude longitude\",task:\"Find an H3 cell\",recipe:true},\n {id:\"jev\",name:\"Jev\",source:\"Community\",purpose:\"Classify text with AI\",description:\"Route selected text to a team using a provider-backed SQL function.\",tags:\"ai classify text sentiment language billing routing\",task:\"Route support requests\",recipe:true,provider:true},\n {id:\"fts\",name:\"Full-text search\",source:\"Core\",purpose:\"Search text in your data\",description:\"Build and query a full-text search index.\",tags:\"search document index fts\",recipe:false},\n {id:\"inet\",name:\"INET\",source:\"Core\",purpose:\"Work with IP addresses\",description:\"Use DuckDB IP address types and functions.\",tags:\"network ip address ipv4 ipv6\",recipe:false},\n {id:\"json\",name:\"JSON\",source:\"Core\",purpose:\"Read and query JSON\",description:\"Read JSON data and transform nested values.\",tags:\"json nested document file\",recipe:false}\n];\nexport const scenarios=[\n [\"happy\",\"First-time use\"],[\"stopped\",\"Kernel stopped\"],[\"installed\",\"Already installed\"],\n [\"failure\",\"Installation fails\"],[\"credentials\",\"Provider credentials\"],[\"incompatible\",\"Unavailable build\"],\n [\"remote\",\"Remote runtime\"],[\"no_recipe\",\"No starter recipe\"],[\"sql_installed\",\"Installed through SQL\"]\n];\nexport function initial(){\n return {tab:\"prototype\",stage:\"browse\",selected:\"excel\",query:\"\",filter:\"all\",inventory:false,scenario:\"happy\",\n runtime:\"running\",target:\"local\",installed:{},loaded:{},desired:{},credential:false,progress:0,\n failedOnce:false,notice:\"\",draft:null,edited:false,remove:false,\n form:{file:\"sales_october.xlsx\",sheet:\"Sales\",url:\"https://example.com/sales.parquet\",lat:\"37.7887987\",lng:\"-122.3931578\",resolution:\"9\",table:\"feedback\",column:\"body\"}};\n}\nexport const entry=s=>entries.find(e=>e.id===s.selected)||entries[0];\nconst quote=v=>String(v).replace(/'/g,\"''\");\nconst ident=v=>'\"'+String(v).replace(/\"/g,'\"\"')+'\"';\nexport function queryFor(s){\n const f=s.form;\n if(s.selected===\"excel\")return \"SELECT *\\nFROM read_xlsx('\"+quote(f.file)+\"',\\n  sheet = '\"+quote(f.sheet)+\"', header = true)\\nLIMIT 5;\";\n if(s.selected===\"httpfs\")return \"SELECT *\\nFROM read_parquet('\"+quote(f.url)+\"')\\nLIMIT 5;\";\n if(s.selected===\"spatial\")return \"SELECT ST_AsText(\\n  ST_Point(\"+Number(f.lng)+\", \"+Number(f.lat)+\")\\n) AS geometry;\";\n if(s.selected===\"h3\")return \"SELECT h3_latlng_to_cell(\\n  \"+Number(f.lat)+\", \"+Number(f.lng)+\", \"+Number(f.resolution)+\"\\n) AS cell_id;\";\n if(s.selected===\"jev\")return \"SELECT jev_choice(t,\\n  'Which team should handle this request?',\\n  ['billing', 'technical', 'sales']) AS team\\nFROM (\\n  SELECT \"+ident(f.column)+\" FROM \"+ident(f.table)+\" LIMIT 5\\n) AS t;\";\n return \"-- \"+entry(s).name+\" is ready in this notebook.\\n-- Add your extension-specific SQL here.\";\n}\nexport function validate(s){\n const f=s.form;\n if(s.selected===\"excel\"&&(!f.file.toLowerCase().endsWith(\".xlsx\")||!f.sheet.trim()))return \"Choose an .xlsx file and a worksheet.\";\n if(s.selected===\"httpfs\"&&!/^https?:\\/\\/.+/.test(f.url))return \"Enter a complete http or https URL.\";\n if([\"h3\",\"spatial\"].includes(s.selected)&&(!f.lat.trim()||!f.lng.trim()||!Number.isFinite(Number(f.lat))||!Number.isFinite(Number(f.lng))||Math.abs(Number(f.lat))>90||Math.abs(Number(f.lng))>180))return \"Use a latitude from -90 to 90 and longitude from -180 to 180.\";\n if(s.selected===\"h3\"&&(!Number.isInteger(Number(f.resolution))||Number(f.resolution)<0||Number(f.resolution)>15))return \"Use an H3 resolution from 0 to 15.\";\n if(s.selected===\"jev\"&&(!f.table.trim()||!f.column.trim()))return \"Choose a table and the text column to share.\";\n return null;\n}\nexport function reduce(s,a){\n const e=entry(s);\n if(a.type===\"tab\")return {...s,tab:a.value};\n if(a.type===\"search\")return {...s,query:a.value};\n if(a.type===\"filter\")return {...s,filter:a.value};\n if(a.type===\"inventory\")return {...s,inventory:a.value};\n if(a.type===\"field\")return {...s,form:{...s.form,[a.key]:a.value},draft:s.edited?s.draft:null,notice:\"\"};\n if(a.type===\"select\")return {...s,selected:a.id,stage:\"configure\",draft:a.id===s.selected?s.draft:null,edited:a.id===s.selected?s.edited:false,notice:\"\",remove:false};\n if(a.type===\"back\")return {...s,stage:\"browse\",notice:\"\",remove:false};\n if(a.type===\"scenario\"){\n  let n={...initial(),scenario:a.value};\n  if(a.value===\"stopped\")n.runtime=\"stopped\";\n  if(a.value===\"installed\")n={...n,stage:\"configure\",installed:{excel:true}};\n  if(a.value===\"failure\")n.stage=\"configure\";\n  if(a.value===\"credentials\")n={...n,selected:\"jev\",stage:\"configure\"};\n  if(a.value===\"incompatible\")n={...n,selected:\"h3\",stage:\"configure\"};\n  if(a.value===\"remote\")n={...n,target:\"remote\",stage:\"configure\"};\n  if(a.value===\"no_recipe\")n={...n,selected:\"inet\",stage:\"configure\"};\n  if(a.value===\"sql_installed\")n={...n,selected:\"h3\",stage:\"browse\",inventory:true,installed:{h3:true},loaded:{h3:true}};\n  return n;\n }\n if(a.type===\"local\")return {...s,target:\"local\",notice:\"The current notebook will use managed local DuckDB.\"};\n if(a.type===\"use\"){\n  if(s.target!==\"local\")return {...s,notice:\"Choose a managed local runtime to prepare this notebook.\"};\n  if(s.scenario===\"incompatible\")return {...s,notice:\"This scenario has no matching build. Choose another runtime or return to the catalog.\"};\n  const problem=validate(s);if(problem)return {...s,notice:problem};\n  return {...s,stage:\"preparing\",progress:0,notice:\"\",desired:{...s.desired,[s.selected]:true}};\n }\n if(a.type===\"tick\"&&s.stage===\"preparing\"){\n  if(s.progress===0)return {...s,runtime:\"running\",progress:1};\n  if(s.progress===1){\n   if(s.scenario===\"failure\"&&!s.failedOnce)return {...s,stage:\"failed\",failedOnce:true,notice:\"The package could not be downloaded. Your task and inputs are saved.\"};\n   return {...s,installed:{...s.installed,[s.selected]:true},progress:2};\n  }\n  if(s.progress===2&&e.provider&&!s.credential)return {...s,stage:\"credentials\"};\n  if(s.progress===2)return {...s,progress:3};\n  return {...s,stage:\"query\",loaded:{...s.loaded,[s.selected]:true},draft:s.draft??queryFor(s),notice:s.edited?\"Your edited query has been preserved.\":\"Ready in this notebook. Review the query before running it.\"};\n }\n if(a.type===\"retry\")return {...s,stage:\"preparing\",progress:0,notice:\"\"};\n if(a.type===\"credential\")return {...s,credential:true,stage:\"preparing\",progress:3,notice:\"\"};\n if(a.type===\"edit\")return {...s,draft:a.value,edited:true};\n if(a.type===\"run\"){\n  if(!s.loaded[s.selected]||s.runtime!==\"running\"||s.target!==\"local\")return {...s,notice:\"The runtime changed. Prepare this notebook again before running.\"};\n  return {...s,stage:\"result\",notice:s.edited||!e.recipe?\"The edited query is preserved. This design prototype does not execute custom SQL.\":\"Sample result shown. No query or provider request was executed.\"};\n }\n if(a.type===\"restart\")return {...s,runtime:\"stopped\",loaded:{},stage:s.draft?\"query\":s.stage,notice:\"Kernel restarted. The query and cached installation remain. Prepare again to restore readiness.\"};\n if(a.type===\"query\")return {...s,stage:\"query\"};\n if(a.type===\"remove\")return {...s,remove:!s.remove};\n if(a.type===\"confirm_remove\")return {...s,desired:{...s.desired,[s.selected]:false},remove:false,notice:\"Automatic preparation removed. The extension can remain loaded until this kernel restarts.\"};\n return s;\n}\nconst el=(tag,props,...children)=>h(tag,props,...children);\nconst button=(label,onClick,props={})=>el(\"button\",{type:\"button\",onClick,...props},label);\nconst badge=(label,cls=\"\")=>el(\"span\",{class:\"badge \"+cls},label);\nfunction Field({label,value,onChange,type=\"text\",hint}){\n return el(\"label\",{class:\"field\"},el(\"span\",null,label),el(\"input\",{type,value,onInput:ev=>onChange(ev.currentTarget.value),autoComplete:\"off\"}),hint&&el(\"small\",null,hint));\n}\nconst docsFor=e=>({\n excel:\"https://duckdb.org/docs/1.5/core_extensions/excel\",\n httpfs:\"https://duckdb.org/docs/1.5/core_extensions/httpfs/overview\",\n spatial:\"https://duckdb.org/docs/1.5/core_extensions/spatial/overview\",\n h3:\"https://duckdb.org/community_extensions/extensions/h3\",\n jev:\"https://duckdb.org/community_extensions/extensions/jev\",\n fts:\"https://duckdb.org/docs/1.5/core_extensions/full_text_search\",\n inet:\"https://duckdb.org/docs/1.5/core_extensions/inet\",\n json:\"https://duckdb.org/docs/1.5/data/json/overview\"\n})[e.id];\nconst stages=[\"Discover\",\"Choose a task\",\"Bind inputs\",\"Prepare notebook\",\"Review query\",\"Get results\",\"Return & recover\"];\nexport function Journey(){\n const rows=[\n [\"Discover\",\"I know what I want to do.\",\"Search task, package or function. Filter Core / Community without an endorsement gate.\",\"Search intent and scroll position survive Back.\"],\n [\"Choose a task\",\"Is this the right extension?\",\"Lead with purpose, inputs and expected output. Keep documentation and functions available.\",\"An absent recipe still allows generic installation.\"],\n [\"Bind inputs\",\"How does it work with my data?\",\"Pick a file, URL or table and columns. Bind values into an editable query.\",\"Validation is inline; preserve valid inputs.\"],\n [\"Prepare notebook\",\"Make this work here.\",\"One action starts the kernel, checks compatibility, uses cache, installs if needed, loads and verifies.\",\"Failure offers Retry; cached installation is reused.\"],\n [\"Review query\",\"Show me what will run.\",\"Focus the new SQL cell, show selected data and any provider or network effect.\",\"Preparing a cell does not execute its task.\"],\n [\"Get results\",\"Did it work?\",\"Run explicitly, show a bounded preview, then let the user edit or keep querying.\",\"An empty result is a result; errors keep the SQL and inputs.\"],\n [\"Return & recover\",\"Will it still work later?\",\"Observe actual runtime inventory, including SQL-installed packages. Save dependency intent with notebook.\",\"Restart invalidates loaded readiness, preserves cells and inputs.\"]\n ];\n return el(\"section\",{class:\"document\"},el(\"p\",{class:\"eyebrow\"},\"COMPLETE JOURNEY\"),el(\"h2\",null,\"From a task to a result\"),el(\"p\",{class:\"intro\"},\"Keep the notebook, the chosen task and the selected data in one continuous flow.\"),\n el(\"div\",{class:\"journey-line\"},stages.map((name,i)=>el(\"span\",{key:name},el(\"b\",null,String(i+1).padStart(2,\"0\")),name))),\n el(\"div\",{class:\"table-scroll\"},el(\"table\",{class:\"journey-table\"},el(\"thead\",null,el(\"tr\",null,[\"Stage\",\"User question\",\"Proposed interaction\",\"Continuity\"].map(t=>el(\"th\",null,t)))),el(\"tbody\",null,rows.map(r=>el(\"tr\",null,r.map((t,i)=>el(i?\"td\":\"th\",null,t))))))),\n el(\"h3\",null,\"Recovery stays in the same task\"),\n el(\"div\",{class:\"recovery-grid\"},[\n [\"Stopped kernel\",\"Start it as part of preparation. Explain the stage in the progress view.\"],\n [\"Failed download\",\"Retry in place with the file, task and query still selected.\"],\n [\"Missing credentials\",\"Pause for secure credential setup, then resume. The prototype uses a demo credential.\"],\n [\"Unavailable build\",\"Name the version/platform issue. Offer a suitable runtime or another extension.\"],\n [\"Remote or custom runtime\",\"Show inspection and manual SQL guidance. Require an explicit switch for managed setup.\"],\n [\"Restart or runtime switch\",\"Discard old readiness. Recheck the selected runtime before enabling Run.\"],\n [\"No starter recipe\",\"Install and open an editable SQL cell with documentation. Do not invent a result.\"],\n [\"Remove notebook dependency\",\"Stop future automatic preparation. Explain that loaded code can remain until restart.\"]\n ].map(([title,body])=>el(\"article\",null,el(\"h4\",null,title),el(\"p\",null,body)))),\n el(\"h3\",null,\"A notebook that can be reopened\"),el(\"p\",null,\"Proposed dependency metadata records package identity, source and preparation intent. Reopening checks the current runtime and restores missing setup only when authorized. Credentials stay in secure storage and are never embedded in shared notebooks. Already installed packages reuse compatible cached artifacts.\"));\n}\nexport function Rules({evidence}){\n return el(\"section\",{class:\"document\"},el(\"p\",{class:\"eyebrow\"},\"WORKFLOW CONTRACT\"),el(\"h2\",null,\"Automation with explicit execution\"),\n el(\"div\",{class:\"rule-grid\"},[\n [\"One preparation action\",\"Start or reuse the current local kernel, resolve a compatible package, install if needed, load and verify. Keep manual diagnostics under Details.\"],\n [\"Readiness follows the runtime\",\"An installed binary is distinct from code loaded in the active connection. Restart and runtime changes invalidate the previous ready state.\"],\n [\"Recipes are optional\",\"Every catalog entry retains an installation path. A task recipe adds input binding and a useful example, without becoming an eligibility filter.\"],\n [\"Run means run\",\"Viewing a task and preparing a query do not execute it. Provider-backed runs disclose selected columns and possible charges before the explicit action.\"]\n ].map(([t,b])=>el(\"article\",null,el(\"h3\",null,t),el(\"p\",null,b)))),\n el(\"h3\",null,\"Bounded solver checks\"),el(\"p\",null,\"The workflow catalog checks our declared finite transitions. It does not inspect DuckDB, prove UI usability or measure time. Bounds equal the length of each supplied trace; they are not click counts.\"),\n el(\"div\",{class:\"table-scroll\"},el(\"table\",null,el(\"thead\",null,el(\"tr\",null,[\"Scenario\",\"Expected behavior\",\"Solver outcome\",\"Receipt\"].map(t=>el(\"th\",null,t)))),el(\"tbody\",null,evidence.map(r=>el(\"tr\",{key:r.solve_id},el(\"td\",null,r.name),el(\"td\",null,r.expected===\"pass\"?\"Allow modeled path\":\"Reject modeled path\"),el(\"td\",null,r.status+\" / \"+r.outcome),el(\"td\",{class:\"mono small\"},r.solve_id)))))),\n el(\"h3\",null,\"Minimal task recipe\"),el(\"pre\",{class:\"code-block\"},\"Task: Read an Excel worksheet\\nInputs: file (.xlsx), sheet, optional header setting\\nRequires: excel, current DuckDB runtime\\nPrepare: install if needed, load, verify\\nOutput: editable SQL cell + expected tabular preview\\nExecute: explicit Run preview\\nRecovery: preserve inputs and SQL; retry preparation\"),\n el(\"p\",null,\"Recipe fields also identify local, network or provider execution, selected data columns, documentation provenance and version requirements. For Jev, a subquery limits the rows and columns passed as the row argument.\"));\n}\nexport function Review(){\n const rows=[\n [\"Most entries stop at copied installation SQL\",\"Provide the shared Use in notebook action and real runtime inventory.\"],\n [\"Guided setup asks users to manage technical stages\",\"Orchestrate preparation and expose only actionable blockers.\"],\n [\"Function reference precedes a useful task\",\"Lead with starter tasks and input selection. Keep the function list one level deeper.\"],\n [\"The HTTPFS example uses a placeholder URL\",\"Bind a selected URL and retain the user's value on retry.\"],\n [\"SQL-installed packages disappear from managed inventory\",\"Reconcile observed runtime state with catalog metadata by source-qualified identity.\"],\n [\"Back or restart can interrupt progress\",\"Retain search, inputs and edited SQL; invalidate readiness separately.\"]\n ];\n return el(\"section\",{class:\"document\"},el(\"p\",{class:\"eyebrow\"},\"DESIGN REVIEW\"),el(\"h2\",null,\"What should change in the product\"),el(\"p\",{class:\"intro\"},\"This notebook specifies a proposed journey. The application code was evaluated in the preceding source review.\"),\n el(\"table\",null,el(\"thead\",null,el(\"tr\",null,el(\"th\",null,\"Observed gap\"),el(\"th\",null,\"Design response\"))),el(\"tbody\",null,rows.map(r=>el(\"tr\",null,r.map(t=>el(\"td\",null,t)))))),\n el(\"h3\",null,\"Implementation order\"),el(\"ol\",{class:\"spaced\"},[\n \"Shared Core/Community installation with version-aware resolution and observed runtime inventory.\",\n \"One preparation action with stopped-kernel startup, progress, cancellation and recovery.\",\n \"Input-bound recipes for Excel, HTTPFS and Spatial, followed by Community and provider examples.\",\n \"Notebook dependency persistence, restart recovery and SQL-installed package reconciliation.\"\n ].map(t=>el(\"li\",null,t))),\n el(\"h3\",null,\"Walkthrough acceptance criteria\"),el(\"ul\",{class:\"spaced\"},[\n \"A first-time user reaches the Excel sample result without copying installation SQL or leaving the notebook.\",\n \"An already installed package uses the cache and loads in the current runtime.\",\n \"Failure, Back and restart retain selected task inputs and edited SQL.\",\n \"Every sample Core/Community entry is discoverable; absent recipes do not block installation.\",\n \"An unavailable build is explained without claiming the extension is generally unsupported.\",\n \"No task query or provider request occurs during browsing or preparation.\",\n \"Ready refers to the current runtime; restarting requires preparation before Run.\",\n \"The full catalog, a narrow sidebar, keyboard focus and real runtime failure states need production usability testing.\"\n ].map(t=>el(\"li\",null,t))),\n el(\"h3\",null,\"Review prompts\"),el(\"p\",null,\"Try Excel with an installation failure. Then try Jev with missing credentials, H3 after a restart, and INET without a recipe. At every stage ask: Is the next action clear? Is the current notebook clear? Are my inputs still here?\"));\n}\nfunction Catalog({s,act}){\n const list=entries.filter(e=>(s.filter===\"all\"||e.source.toLowerCase()===s.filter)&&(!s.inventory||s.installed[e.id])&&\n s.query.toLowerCase().trim().split(/\\s+/).every(q=>(e.name+\" \"+e.purpose+\" \"+e.tags).toLowerCase().includes(q)));\n return el(\"aside\",{class:\"catalog\"},el(\"div\",{class:\"catalog-head\"},el(\"h2\",null,\"Extensions\"),el(\"p\",{class:\"muted\"},\"Find a tool for your next query.\")),\n el(\"div\",{class:\"segmented\"},button(\"Browse\",()=>act({type:\"inventory\",value:false}),{\"aria-pressed\":!s.inventory}),button(\"In this notebook\",()=>act({type:\"inventory\",value:true}),{\"aria-pressed\":s.inventory})),\n el(\"label\",{class:\"search\"},el(\"span\",{class:\"sr-only\"},\"Search extensions or tasks\"),el(\"input\",{placeholder:\"Search tasks or extensions\",value:s.query,onInput:ev=>act({type:\"search\",value:ev.currentTarget.value})})),\n el(\"div\",{class:\"filters\"},[[\"all\",\"All\"],[\"core\",\"Core\"],[\"community\",\"Community\"]].map(([id,label])=>button(label,()=>act({type:\"filter\",value:id}),{\"aria-pressed\":s.filter===id}))),\n el(\"div\",{class:\"catalog-list\"},list.length?list.map(e=>button(el(\"span\",null,el(\"span\",{class:\"entry-top\"},el(\"strong\",null,e.name),badge(e.source)),el(\"span\",{class:\"entry-purpose\"},e.purpose),s.installed[e.id]&&el(\"span\",{class:\"entry-state\"},s.loaded[e.id]?\"Loaded in this runtime\":\"Installed, preparation needed\")),()=>act({type:\"select\",id:e.id}),{class:\"entry \"+(s.stage!==\"browse\"&&s.selected===e.id?\"selected\":\"\"),\"data-entry\":e.id})):el(\"div\",{class:\"empty\"},el(\"strong\",null,s.inventory?\"No matching installed packages\":\"No matching extensions\"),el(\"p\",null,s.inventory?\"Browse to add an extension to this notebook.\":\"Try a task such as Excel, map or classify.\"),button(\"Clear search and filters\",()=>{act({type:\"search\",value:\"\"});act({type:\"filter\",value:\"all\"});}))),\n el(\"p\",{class:\"catalog-foot\"},s.inventory?\"Observed installation and load state, simulated here. SQL-installed packages are included.\":\"8 sample entries in this prototype. The product uses the full unique Core and Community catalog.\"));\n}\nfunction Inputs({s,act}){\n const field=(key,label,hint,type)=>h(Field,{key,label,hint,type,value:s.form[key],onChange:value=>act({type:\"field\",key,value})});\n if(s.selected===\"excel\")return el(\"div\",{class:\"form-grid\"},field(\"file\",\"Excel file\",\"Sample file name. No file is opened by this prototype.\"),field(\"sheet\",\"Worksheet\",\"Use the worksheet name from your file.\"));\n if(s.selected===\"httpfs\")return el(\"div\",{class:\"form-grid single\"},field(\"url\",\"Parquet URL\",\"The preview is simulated. example.com is a placeholder, not a working dataset.\"));\n if([\"spatial\",\"h3\"].includes(s.selected))return el(\"div\",{class:\"form-grid\"},field(\"lat\",\"Latitude\",null,\"number\"),field(\"lng\",\"Longitude\",null,\"number\"),s.selected===\"h3\"&&field(\"resolution\",\"H3 resolution\",\"0-15. Higher values describe smaller cells.\",\"number\"));\n if(s.selected===\"jev\")return el(\"div\",{class:\"form-grid\"},field(\"table\",\"Source table\",\"Sample table: feedback\"),field(\"column\",\"Text column\",\"Only this column is passed to the provider.\"));\n return null;\n}\nfunction ResultTable({s}){\n let cols,rows;\n if(s.selected===\"excel\"||s.selected===\"httpfs\"){cols=[\"region\",\"product\",\"revenue\"];rows=[[\"North\",\"Notebook\",1240],[\"South\",\"Pencil\",380],[\"West\",\"Notebook\",920],[\"North\",\"Marker\",260],[\"East\",\"Pencil\",410]];}\n if(s.selected===\"spatial\"){cols=[\"geometry\"];rows=[[\"POINT (\"+s.form.lng+\" \"+s.form.lat+\")\"]];}\n if(s.selected===\"h3\"){cols=[\"cell_id\"];rows=[[\"demo_h3_cell_id\"]];}\n if(s.selected===\"jev\"){cols=[\"team\"];rows=[[\"billing\"],[\"technical\"],[\"sales\"]];}\n return el(\"div\",{class:\"table-scroll result-table\"},el(\"table\",null,el(\"thead\",null,el(\"tr\",null,cols.map(c=>el(\"th\",null,c)))),el(\"tbody\",null,rows.map(r=>el(\"tr\",null,r.map(c=>el(\"td\",null,String(c))))))));\n}\nfunction Canvas({s,act}){\n const e=entry(s),ready=!!s.loaded[s.selected]&&s.runtime===\"running\";\n if(s.stage===\"browse\")return el(\"main\",{class:\"canvas\"},el(\"div\",{class:\"welcome\"},el(\"p\",{class:\"eyebrow\"},\"START WITH YOUR TASK\"),el(\"h1\",null,\"What would you like to query?\"),el(\"p\",{class:\"intro\"},\"Choose a purpose. Jute prepares DuckDB in this notebook and gives you a query to review.\"),\n el(\"div\",{class:\"task-grid\"},entries.filter(e=>e.recipe).slice(0,4).map(e=>button(el(\"span\",null,el(\"strong\",null,e.purpose),el(\"span\",{class:\"muted\"},e.name+\" · \"+e.source)),()=>act({type:\"select\",id:e.id}),{class:\"task-tile\"}))),\n el(\"div\",{class:\"notebook-placeholder\"},el(\"span\",{class:\"cell-number\"},\"SQL\"),el(\"p\",null,\"Your prepared query will appear here.\")),\n el(\"details\",{class:\"baseline\"},el(\"summary\",null,\"Read the complete journey\"),el(\"ol\",null,stages.map(t=>el(\"li\",null,t))),el(\"p\",{class:\"muted\"},\"Enable active content on this output to explore the interactive paths.\"))));\n const disabled=s.scenario===\"incompatible\"||s.target!==\"local\";\n return el(\"main\",{class:\"canvas\"},el(\"div\",{class:\"detail-top\"},button(\"← Extensions\",()=>act({type:\"back\"}),{class:\"text-button\"}),el(\"div\",{class:\"badges\"},badge(e.source),ready?badge(\"Ready here\"):s.installed[e.id]?badge(\"Installed\"):null)),\n el(\"div\",{class:\"detail-title\"},el(\"p\",{class:\"eyebrow\"},e.name.toUpperCase()),el(\"h1\",null,e.purpose),el(\"p\",{class:\"intro\"},e.description)),\n [\"configure\",\"credentials\",\"preparing\",\"failed\"].includes(s.stage)&&el(\"div\",{class:\"task-summary\"},el(\"div\",null,el(\"span\",{class:\"small muted\"},\"SELECTED TASK\"),el(\"h3\",null,e.recipe?e.task:\"Install and write your own query\")),el(\"span\",{class:\"small muted\"},e.recipe?\"Editable SQL + preview\":\"Generic installation available\")),\n s.stage===\"configure\"&&el(\"div\",{class:\"configure\"},!e.recipe&&el(\"div\",{class:\"notice\"},el(\"strong\",null,\"No starter recipe yet\"),el(\"p\",null,\"Prepare the extension in this notebook, then use its documentation to write a query.\")),\n h(Inputs,{s,act}),\n s.scenario===\"incompatible\"&&el(\"div\",{class:\"notice\"},el(\"strong\",null,\"No matching build in this scenario\"),el(\"p\",null,\"The chosen package has no available build for the simulated runtime. This does not mean the extension is generally unsupported.\"),button(\"Return to catalog\",()=>act({type:\"back\"}))),\n s.target!==\"local\"&&el(\"div\",{class:\"notice\"},el(\"strong\",null,\"Remote runtime selected\"),el(\"p\",null,\"Managed preparation is available on local DuckDB. Keep manual SQL guidance for a runtime you manage.\"),button(\"Use managed local runtime\",()=>act({type:\"local\"}))),\n el(\"div\",{class:\"action-row\"},button(e.recipe?\"Use in notebook\":\"Install in notebook\",()=>act({type:\"use\"}),{class:\"primary\",disabled,\"data-action\":\"use\"}),el(\"span\",{class:\"muted small\"},e.provider?\"Provider calls happen only when you run the query.\":\"Prepares the extension and SQL. The task runs when you choose Run.\")),\n el(\"details\",{class:\"advanced\"},el(\"summary\",null,\"Installation SQL and documentation\"),el(\"pre\",{class:\"code-block\"},\"INSTALL \"+e.id+\" FROM \"+e.source.toLowerCase()+\";\\nLOAD \"+e.id+\";\"),el(\"p\",{class:\"small muted\"},\"The product resolves builds for the selected DuckDB version and platform. Browse and copying are passive.\"),el(\"p\",null,el(\"a\",{href:docsFor(e),target:\"_blank\",rel:\"noopener noreferrer\",\"data-documentation\":e.id},\"Open DuckDB documentation\")))),\n s.stage===\"preparing\"&&el(\"section\",{\"aria-live\":\"polite\",class:\"prepare\"},el(\"h2\",null,\"Preparing this notebook\"),el(\"p\",{class:\"muted\"},\"Stay here. Your task and inputs are saved.\"),\n el(\"ol\",{class:\"progress\"},[\n s.runtime===\"stopped\"?\"Start the DuckDB kernel\":\"Connect to the DuckDB kernel\",\n s.installed[e.id]?\"Reuse compatible cached installation\":\"Install \"+e.name,\n e.provider?\"Check credentials\":\"Load the extension\",\n \"Verify the current runtime\"\n ].map((label,i)=>el(\"li\",{class:i<s.progress?\"complete\":i===s.progress?\"current\":\"\",key:i},el(\"span\",{class:\"step-number\"},i<s.progress?\"✓\":String(i+1)),el(\"span\",null,label),i===s.progress&&el(\"small\",null,\"Working\")))),\n el(\"p\",{class:\"small muted\"},\"Simulated progress. No package is downloaded.\")),\n s.stage===\"failed\"&&el(\"section\",{class:\"notice\"},el(\"h2\",null,\"Download interrupted\"),el(\"p\",null,\"The extension could not be downloaded. Your selected task, file and worksheet are still here.\"),el(\"div\",{class:\"action-row\"},button(\"Retry preparation\",()=>act({type:\"retry\"}),{class:\"primary\",\"data-action\":\"retry\"}),button(\"Review inputs\",()=>act({type:\"select\",id:s.selected}))),el(\"details\",null,el(\"summary\",null,\"Technical details\"),el(\"p\",{class:\"mono small\"},\"Simulated network failure during installation. The next attempt will succeed.\"))),\n s.stage===\"credentials\"&&el(\"section\",{class:\"credential-box\"},el(\"h2\",null,\"Connect Jev to continue\"),el(\"p\",null,\"A TypeSafe credential is needed before this extension can be prepared for provider-backed queries.\"),el(\"div\",{class:\"credential-demo\"},el(\"span\",null,\"TypeSafe API token\"),el(\"code\",null,\"•••• •••• demo credential\")),el(\"p\",{class:\"small muted\"},\"This prototype uses a demo credential. Product credentials belong in secure storage, outside SQL and notebook files.\"),button(\"Use demo credential and continue\",()=>act({type:\"credential\"}),{class:\"primary\",\"data-action\":\"credential\"}),el(\"p\",{class:\"small muted\"},\"This step does not send table rows or run inference.\")),\n [\"query\",\"result\"].includes(s.stage)&&el(\"section\",{class:\"query-section\"},\n !ready&&el(\"div\",{class:\"notice\"},el(\"strong\",null,\"Prepare again after restart\"),el(\"p\",null,\"The query and installed package are preserved. Readiness must be renewed for this kernel.\"),button(\"Prepare again\",()=>act({type:\"use\"}),{class:\"primary\",\"data-action\":\"restore\"})),\n el(\"div\",{class:\"cell-toolbar\"},el(\"div\",null,el(\"strong\",null,\"SQL\"),el(\"span\",{class:\"small muted\"},\" · \"+e.name+\" · current notebook\")),badge(ready?\"Ready here\":\"Preparation needed\")),\n el(\"label\",{class:\"sr-only\",for:\"query-editor\"},\"Editable SQL query\"),\n el(\"textarea\",{id:\"query-editor\",class:\"query-editor\",value:s.draft||\"\",spellcheck:false,onInput:ev=>act({type:\"edit\",value:ev.currentTarget.value})}),\n e.provider&&el(\"div\",{class:\"provider-notice\"},el(\"strong\",null,\"Before running\"),el(\"p\",null,\"The selected \"+s.form.column+\" values from \"+s.form.table+\" would be sent to TypeSafe. API charges may apply. The query limits the shared input to 5 rows.\")),\n s.selected===\"httpfs\"&&el(\"p\",{class:\"small muted\"},\"Running this query in the product requests the selected URL. This prototype uses sample rows.\"),\n el(\"div\",{class:\"action-row\"},button(e.recipe?\"Run preview\":\"Review custom query\",()=>act({type:\"run\"}),{class:\"primary\",disabled:!ready||(!e.recipe&&!s.edited),\"data-action\":\"run\"}),button(\"Edit inputs\",()=>act({type:\"select\",id:s.selected})),el(\"span\",{class:\"small muted\"},\"Sample preview. SQL is not executed.\")),\n s.stage===\"result\"&&el(\"section\",{class:\"results\",\"aria-live\":\"polite\"},el(\"div\",{class:\"result-heading\"},el(\"h2\",null,s.edited||!e.recipe?\"Your query is preserved\":\"First result\"),badge(\"Sample data\")),\n s.edited||!e.recipe?el(\"p\",null,\"Custom SQL is not executed by this prototype. Your edits remain in the cell.\"):h(ResultTable,{s}),\n el(\"p\",{class:\"small muted\"},s.selected===\"h3\"?\"The H3 index is an illustrative sample value. The implementation must display the actual query output.\":\"Illustrative sample output. No file was opened and no remote request was made.\"),\n el(\"div\",{class:\"action-row\"},button(\"Keep querying\",()=>act({type:\"query\"})),button(\"Browse extensions\",()=>act({type:\"back\"})))),\n el(\"details\",{class:\"advanced\"},el(\"summary\",null,\"Notebook dependency and lifecycle\"),el(\"p\",null,\"Proposed behavior: retain this package and source as a notebook dependency. Reopen and restart recheck readiness on the current runtime.\"),\n button(\"Remove automatic preparation\",()=>act({type:\"remove\"})),\n s.remove&&el(\"div\",{class:\"notice\"},el(\"p\",null,\"Remove \"+e.name+\" from automatic notebook preparation? Existing cells stay. Loaded code can remain until the kernel restarts.\"),button(\"Remove from preparation\",()=>act({type:\"confirm_remove\"})),button(\"Keep dependency\",()=>act({type:\"remove\"}))))),\n s.notice&&el(\"p\",{class:\"status-note\",role:\"status\"},s.notice));\n}\nexport function View({s,act=()=>{},evidence=[]}){\n return el(\"div\",{class:\"design-root\"},\n el(\"header\",{class:\"design-header\"},el(\"div\",null,el(\"p\",{class:\"eyebrow\"},\"JUTE · INTERACTIVE DESIGN\"),el(\"h1\",null,\"Extensions, with a purpose\"),el(\"p\",{class:\"muted\"},\"Discover a task, prepare DuckDB, get your first result.\")),el(\"span\",{class:\"prototype-label\"},\"Prototype · sample data\")),\n el(\"nav\",{class:\"top-tabs\",\"aria-label\":\"Design sections\"},[\n [\"prototype\",\"Try the journey\"],[\"map\",\"Full journey\"],[\"rules\",\"Rules & solver\"],[\"review\",\"Product review\"]\n ].map(([id,label])=>button(label,()=>act({type:\"tab\",value:id}),{\"aria-current\":s.tab===id?\"page\":undefined}))),\n s.tab===\"prototype\"?el(\"section\",{class:\"prototype\"},\n el(\"div\",{class:\"scenario-bar\"},el(\"label\",null,\"Explore a scenario\",el(\"select\",{value:s.scenario,onChange:ev=>act({type:\"scenario\",value:ev.currentTarget.value})},scenarios.map(([id,title])=>el(\"option\",{value:id},title)))),\n el(\"span\",{class:\"small muted\"},\"All setup and results are simulated.\"),button(\"Reset\",()=>act({type:\"scenario\",value:s.scenario}),{class:\"text-button\"})),\n el(\"div\",{class:\"app-frame\"},el(\"div\",{class:\"app-toolbar\"},el(\"div\",null,el(\"strong\",null,\"Jute\"),el(\"span\",{class:\"muted\"},\" / Revenue exploration.ipynb\")),el(\"div\",{class:\"runtime\"},el(\"span\",null,s.target===\"local\"?\"Local DuckDB 1.5.5\":\"Remote DuckDB\"),badge(s.runtime===\"running\"?\"Kernel running\":\"Kernel stopped\"),button(\"Restart kernel\",()=>act({type:\"restart\"}),{class:\"small\"}))),\n el(\"div\",{class:\"app-body\"},h(Catalog,{s,act}),h(Canvas,{s,act}))),\n el(\"div\",{class:\"design-caption\"},el(\"strong\",null,\"Design intent\"),el(\"span\",null,\"One preparation action handles setup. The user chooses when to run. Search, input values and edited SQL survive recovery.\"))):\n s.tab===\"map\"?h(Journey,{}):s.tab===\"rules\"?h(Rules,{evidence}):h(Review,{}),\n el(\"footer\",{class:\"design-footer\"},el(\"span\",null,\"Full unique Core and Community catalog in the product. Eight representative entries here.\"),el(\"span\",null,\"Enable active content for interactive controls.\")));\n}\n";
 const clientSource = "\nimport { h, hydrate, render } from \"https://esm.sh/preact@10.24.3\";\nimport { View, initial, reduce } from \"./view.ts\";\nconst root=document.getElementById(\"app\");\nlet state=initial();\nlet timer;\nconst evidence=JSON.parse(document.getElementById(\"evidence-data\").textContent);\nfunction paint(){render(h(View,{s:state,act,evidence}),root);}\nfunction act(action){\n clearTimeout(timer);\n state=reduce(state,action);\n paint();\n if(state.stage===\"preparing\")timer=setTimeout(()=>act({type:\"tick\"}),window.__JOURNEY_TEST_MS__||480);\n if(action.type===\"tick\"&&state.stage===\"query\")document.getElementById(\"query-editor\")?.focus({preventScroll:true});\n}\nhydrate(h(View,{s:state,act,evidence}),root);\ndocument.documentElement.dataset.interactive=\"true\";\n";
 const css = "\n:root{--bg:oklch(99% .002 240);--surface:oklch(100% 0 0);--fg:oklch(18% .012 250);--muted:oklch(48% .012 250);--border:oklch(89% .005 250);--accent:oklch(52% .18 255);--soft:oklch(97% .003 240);color-scheme:light;font-family:-apple-system,BlinkMacSystemFont,\"SF Pro Text\",system-ui,sans-serif;color:var(--fg);background:var(--bg)}\n*{box-sizing:border-box}body{margin:0}button,input,select,textarea{font:inherit}button{border:1px solid var(--border);background:var(--surface);border-radius:6px;color:var(--fg);padding:9px 13px;min-height:40px;cursor:pointer;line-height:1.3}button:hover:not(:disabled){background:var(--soft);border-color:var(--muted)}button:disabled{opacity:.5;cursor:not-allowed}button:focus-visible,input:focus-visible,select:focus-visible,textarea:focus-visible,summary:focus-visible{outline:2px solid var(--accent);outline-offset:3px}a{color:var(--accent)}h1,h2,h3,h4,p{margin:0}h1{font-size:30px;letter-spacing:-.03em;line-height:1.12;font-weight:650}h2{font-size:23px;letter-spacing:-.025em;line-height:1.2}h3{font-size:16px;line-height:1.3;font-weight:650}h4{font-size:14px}p{line-height:1.6}small,.small{font-size:12px}.muted{color:var(--muted)}.mono,code,pre,textarea{font-family:ui-monospace,\"SF Mono\",Menlo,monospace}.eyebrow{font-size:10px;font-weight:700;letter-spacing:.13em;line-height:1.5;color:var(--muted);margin-bottom:11px}.design-root{max-width:1280px;margin:auto;padding:30px 28px 20px}.design-header{display:flex;align-items:flex-start;justify-content:space-between;gap:20px;padding-bottom:25px}.design-header h1{font-size:34px}.design-header p:last-child{margin-top:10px;font-size:14px}.prototype-label{font-size:11px;border:1px solid var(--border);border-radius:4px;padding:7px 10px;white-space:nowrap}.top-tabs{display:flex;gap:22px;border-bottom:1px solid var(--border);margin-bottom:22px;overflow:auto}.top-tabs button{padding:12px 0;border:0;border-bottom:2px solid transparent;border-radius:0;color:var(--muted);background:none;font-size:13px;white-space:nowrap}.top-tabs button[aria-current=page]{border-bottom-color:var(--fg);color:var(--fg);font-weight:650}.scenario-bar{display:flex;align-items:center;gap:16px;margin-bottom:14px;flex-wrap:wrap}.scenario-bar label{font-size:12px;font-weight:600;display:flex;align-items:center;gap:10px}.scenario-bar select{font-size:12px;max-width:210px;border:1px solid var(--border);border-radius:5px;padding:7px 28px 7px 10px;background:var(--surface);color:var(--fg);min-height:38px}.scenario-bar .text-button{margin-left:auto}.app-frame{background:var(--surface);border:1px solid var(--border);border-radius:8px;overflow:hidden}.app-toolbar{display:flex;justify-content:space-between;align-items:center;gap:15px;border-bottom:1px solid var(--border);padding:12px 17px;background:var(--soft);font-size:12px;flex-wrap:wrap}.runtime{display:flex;align-items:center;gap:10px;flex-wrap:wrap}.runtime button{min-height:32px;padding:5px 8px;background:transparent}.badge{display:inline-flex;font-size:10px;border:1px solid var(--border);border-radius:4px;line-height:1.3;padding:4px 6px;color:var(--muted);font-weight:500;white-space:nowrap}.app-body{display:grid;grid-template-columns:275px minmax(0,1fr);min-height:650px}.catalog{border-right:1px solid var(--border);display:flex;flex-direction:column;min-width:0}.catalog-head{padding:21px 18px 14px}.catalog-head h2{font-size:17px;letter-spacing:-.02em}.catalog-head p{font-size:12px;margin-top:5px}.segmented{display:flex;gap:3px;margin:0 15px 14px;background:var(--soft);padding:3px;border-radius:6px}.segmented button{font-size:11px;flex:1;padding:7px 5px;min-height:32px;border-color:transparent;background:transparent}.segmented button[aria-pressed=true]{background:var(--surface);border-color:var(--border);font-weight:600}.search{display:block;margin:0 15px 10px}.search input{width:100%;font-size:12px;padding:10px 11px;border:1px solid var(--border);border-radius:6px;min-height:40px;background:var(--surface)}.filters{display:flex;gap:6px;margin:0 15px 14px}.filters button{min-height:30px;padding:5px 9px;border-color:transparent;font-size:11px;color:var(--muted)}.filters button[aria-pressed=true]{border-color:var(--border);background:var(--soft);color:var(--fg)}.catalog-list{padding:0 8px;flex:1}.entry{display:block;width:100%;text-align:left;border-color:transparent;border-radius:5px;margin-bottom:3px;padding:11px 10px}.entry.selected{background:var(--soft);border-color:var(--border)}.entry-top{display:flex;justify-content:space-between;align-items:center;gap:8px}.entry strong{font-size:13px;font-weight:600}.entry-purpose{display:block;margin-top:5px;font-size:11px;color:var(--muted);line-height:1.5}.entry-state{display:block;margin-top:5px;font-size:10px;font-weight:650}.catalog-foot{font-size:10px;color:var(--muted);padding:20px 17px;line-height:1.6}.canvas{padding:26px 32px 34px;min-width:0}.welcome{padding-top:30px}.intro{font-size:14px;color:var(--muted);max-width:620px;margin-top:14px;line-height:1.65}.task-grid{display:grid;grid-template-columns:1fr 1fr;gap:12px;margin-top:29px}.task-tile{text-align:left;min-height:94px;padding:17px;background:var(--surface)}.task-tile strong{display:block;font-size:13px;margin-bottom:8px}.task-tile .muted{display:block;font-size:11px}.notebook-placeholder{display:flex;align-items:center;gap:17px;padding:24px 0;margin-top:38px;border-top:1px solid var(--border);border-bottom:1px solid var(--border);font-size:12px;color:var(--muted)}.cell-number{font-size:10px;font-family:ui-monospace,monospace}.baseline{margin-top:20px;font-size:12px;color:var(--muted)}summary{cursor:pointer;font-size:12px;line-height:1.5;padding:5px 0}.baseline ol{padding-left:20px;line-height:2}.detail-top{display:flex;justify-content:space-between;align-items:center;gap:10px;margin-bottom:27px}.badges{display:flex;gap:6px}.text-button{padding:5px 0;border:0;background:transparent;min-height:32px;font-size:12px}.detail-title{margin-bottom:28px}.detail-title h1{font-size:29px}.detail-title .intro{font-size:13px;margin-top:10px}.task-summary{display:flex;justify-content:space-between;align-items:center;gap:15px;background:var(--soft);padding:16px 18px;border:1px solid var(--border);border-radius:6px;margin-bottom:24px}.task-summary h3{margin-top:6px}.task-summary .small{font-size:10px}.form-grid{display:grid;grid-template-columns:1fr 1fr;gap:18px}.form-grid.single{grid-template-columns:1fr}.field{display:flex;flex-direction:column;gap:8px;min-width:0}.field>span{font-size:12px;font-weight:600}.field input{width:100%;padding:10px 11px;border:1px solid var(--border);border-radius:5px;font-size:12px;min-height:42px;color:var(--fg);background:var(--surface)}.field small{font-size:10px;color:var(--muted);line-height:1.6}.action-row{display:flex;align-items:center;gap:11px;margin-top:22px;flex-wrap:wrap}.action-row .muted{max-width:270px;font-size:11px;line-height:1.6}.primary{background:var(--accent);color:white;border-color:var(--accent);font-size:12px;font-weight:600;padding:11px 15px}.primary:hover:not(:disabled){background:var(--accent);border-color:var(--accent);filter:brightness(.94)}.advanced{margin-top:26px;padding-top:13px;border-top:1px solid var(--border)}.advanced>p,.advanced>button{margin-top:13px}.code-block{padding:15px;background:var(--soft);border:1px solid var(--border);border-radius:5px;font-size:12px;line-height:1.7;overflow:auto;white-space:pre-wrap;word-break:break-word;margin-top:13px}.notice{border:1px solid var(--border);background:var(--soft);border-radius:6px;padding:17px;margin-bottom:20px;font-size:12px}.notice p{margin-top:7px}.notice button{margin-top:12px}.notice h2{font-size:20px}.prepare{padding:4px 0}.prepare>p{font-size:12px;margin-top:10px}.progress{list-style:none;margin:25px 0;padding:0}.progress li{display:flex;align-items:center;gap:11px;padding:13px 0;border-bottom:1px solid var(--border);color:var(--muted);font-size:12px}.progress .current{color:var(--fg);font-weight:650}.progress .complete{color:var(--fg)}.progress small{margin-left:auto;font-size:10px}.step-number{display:flex;align-items:center;justify-content:center;border:1px solid var(--border);border-radius:50%;width:23px;height:23px;font-size:10px}.credential-box h2{font-size:22px}.credential-box p{font-size:12px;margin:13px 0}.credential-demo{padding:14px;border:1px solid var(--border);border-radius:5px;display:flex;justify-content:space-between;gap:16px;font-size:12px;margin-top:17px}.cell-toolbar{display:flex;align-items:center;justify-content:space-between;gap:12px;border:1px solid var(--border);border-bottom:0;border-radius:5px 5px 0 0;padding:12px;font-size:12px}.query-editor{display:block;resize:vertical;width:100%;min-height:180px;line-height:1.7;font-size:12px;padding:16px;color:var(--fg);border:1px solid var(--border);border-radius:0 0 5px 5px;background:var(--soft);tab-size:2}.provider-notice{padding:14px 0;border-bottom:1px solid var(--border);font-size:12px}.provider-notice p{font-size:11px;color:var(--muted);margin-top:5px}.query-section>p{margin-top:12px}.results{margin-top:29px;padding-top:23px;border-top:1px solid var(--border)}.result-heading{display:flex;justify-content:space-between;align-items:center;margin-bottom:15px}.result-heading h2{font-size:20px}.results>p{font-size:12px;margin:12px 0}.table-scroll{overflow:auto;max-width:100%}table{border-collapse:collapse;width:100%;font-size:12px;line-height:1.65;text-align:left}th,td{border-bottom:1px solid var(--border);padding:11px 13px;vertical-align:top}th{font-weight:650}thead{background:var(--soft)}.result-table td{font-variant-numeric:tabular-nums}.status-note{font-size:11px;border-top:1px solid var(--border);padding-top:14px;margin-top:20px;color:var(--muted)}.design-caption{display:flex;align-items:flex-start;gap:18px;font-size:11px;line-height:1.7;padding:16px 4px;color:var(--muted)}.design-caption strong{color:var(--fg);white-space:nowrap}.design-footer{display:flex;justify-content:space-between;gap:16px;font-size:10px;color:var(--muted);border-top:1px solid var(--border);padding-top:18px;line-height:1.7;margin-top:14px}.document{padding:20px 3px 35px}.document h2{font-size:31px}.document h3{font-size:19px;margin:32px 0 14px}.document>p{font-size:13px;color:var(--muted);max-width:900px}.document table{margin-top:24px}.journey-line{display:flex;margin:28px 0;border-top:1px solid var(--border);border-bottom:1px solid var(--border);overflow:auto}.journey-line>span{min-width:120px;flex:1;font-size:11px;padding:17px 12px;line-height:1.5}.journey-line b{display:block;font-size:10px;color:var(--muted);margin-bottom:7px;font-variant-numeric:tabular-nums}.journey-table{min-width:760px}.recovery-grid,.rule-grid{display:grid;grid-template-columns:1fr 1fr;border-top:1px solid var(--border);border-left:1px solid var(--border);margin-top:20px}.recovery-grid article,.rule-grid article{padding:21px;border-right:1px solid var(--border);border-bottom:1px solid var(--border)}.recovery-grid p,.rule-grid p{font-size:12px;color:var(--muted);margin-top:10px}.rule-grid h3{margin:0;font-size:16px}.spaced{font-size:13px;line-height:1.8;padding-left:22px;max-width:950px}.spaced li{margin-bottom:11px}.empty{padding:25px 10px;font-size:12px}.empty p{font-size:11px;margin:10px 0}.empty button{font-size:11px}.sr-only{position:absolute;width:1px;height:1px;padding:0;margin:-1px;overflow:hidden;clip:rect(0,0,0,0);white-space:nowrap;border:0}\n@media(max-width:900px){.design-root{padding:22px 16px}.app-body{grid-template-columns:235px minmax(0,1fr)}.canvas{padding:24px}.task-grid{grid-template-columns:1fr}.welcome{padding-top:12px}.design-header h1{font-size:29px}.task-summary{display:block}.task-summary>span{display:block;margin-top:10px}}\n@media(max-width:620px){.design-root{padding:17px 12px}.design-header{display:block}.prototype-label{display:inline-block;margin-top:15px}.design-header h1{font-size:27px}.top-tabs{gap:18px}.app-body{display:flex;flex-direction:column;min-height:0}.catalog{border-right:0;border-bottom:1px solid var(--border)}.catalog-list{max-height:245px;overflow:auto}.catalog-foot{padding:12px 17px}.canvas{padding:22px 19px}.catalog-head{padding-top:17px}.task-grid,.form-grid{grid-template-columns:1fr}.app-toolbar{padding:12px}.runtime{gap:7px}.design-caption,.design-footer{display:block}.design-caption strong{display:block;margin-bottom:5px}.design-footer span{display:block;margin-bottom:8px}.scenario-bar{gap:8px}.scenario-bar>span{display:none}.scenario-bar label{flex-wrap:wrap}.scenario-bar .text-button{margin-left:0}.recovery-grid,.rule-grid{grid-template-columns:1fr}.credential-demo{display:block}.credential-demo code{display:block;margin-top:10px}.cell-toolbar{flex-wrap:wrap}.detail-title h1{font-size:26px}.document h2{font-size:27px}.document>table{font-size:11px}.document td,.document th{padding:10px 7px}}\n@media(prefers-reduced-motion:reduce){*{scroll-behavior:auto!important;animation:none!important;transition:none!important}}\n";
 const evidence = [{"name":"First useful result","expected":"pass","outcome":"pass","status":"sat","solve_id":"sol_fecf6b91ad9e4a83"},{"name":"Retry after installation failure","expected":"pass","outcome":"pass","status":"sat","solve_id":"sol_8ada51e285e44a8f"},{"name":"Credential-based explicit use","expected":"pass","outcome":"pass","status":"sat","solve_id":"sol_64c34fda24744ea0"},{"name":"Skip runtime verification","expected":"fail","outcome":"fail","status":"unsat","solve_id":"sol_64f4fcbe1f9b4ef6"},{"name":"Use stale readiness after restart","expected":"fail","outcome":"fail","status":"unsat","solve_id":"sol_bf51a29bfe304dbe"},{"name":"Send provider request during preparation","expected":"fail","outcome":"fail","status":"unsat","solve_id":"sol_e83d287763c64063"}];
 const dir = await Deno.makeTempDir({prefix:"jute-duckdb-journey-"});
 await Deno.writeTextFile(dir+"/view.ts",viewSource);
 await Deno.writeTextFile(dir+"/client.ts",clientSource);
 const views = await import("file://"+dir+"/view.ts");
 const baseline = ssr(h(views.View,{s:views.initial(),evidence}));
 const httpPlugin = {name:"inline-http",setup(b){
  b.onResolve({filter:/^https?:\/\//},a=>({path:a.path,namespace:"http-url"}));
  b.onResolve({filter:/.*/,namespace:"http-url"},a=>({path:new URL(a.path,a.importer).toString(),namespace:"http-url"}));
  b.onLoad({filter:/.*/,namespace:"http-url"},async a=>{
   const response=await fetch(a.path);
   if(!response.ok)throw new Error("Build dependency unavailable: "+response.status);
   return {contents:await response.text(),loader:"js"};
  });
 }};
 const built=await esbuild.build({entryPoints:[dir+"/client.ts"],plugins:[httpPlugin],bundle:true,format:"iife",minify:true,write:false,legalComments:"none",target:"es2020"});
 await esbuild.stop();
 const js=built.outputFiles[0].text.replace(/<\/script/gi,"<\\/script");
 const html="<!doctype html><html lang=\"en\"><head><meta charset=\"utf-8\"><meta name=\"viewport\" content=\"width=device-width, initial-scale=1\"><title>DuckDB extension journey | Jute design</title><style>"+css+"</style></head><body><div id=\"app\">"+baseline+"</div><script id=\"evidence-data\" type=\"application/json\">"+JSON.stringify(evidence)+"</script><script>"+js+"</script></body></html>";
 const tests=[];
 const assert=(name,condition)=>{if(!condition)throw new Error("Verification failed: "+name);tests.push({name,status:"pass"});};
 assert("One self-contained document",!/<script[^>]+src=|<link[^>]+href=|<iframe|<img[^>]+src=/i.test(html));
 assert("No banned separator glyphs",!/[\u2013\u2014]/.test(html)&&!/&(?:mdash|ndash);|&#(?:8211|8212);/.test(html));
 const off=new JSDOM(html,{runScripts:"outside-only"});
 assert("Meaningful scripts-off baseline",off.window.document.querySelectorAll("[data-entry]").length===8&&off.window.document.body.textContent.includes("What would you like to query?")&&off.window.document.body.textContent.includes("Return & recover"));
 let externalAttempts=0;
 const dom=new JSDOM(html,{runScripts:"dangerously",beforeParse(w){w.__JOURNEY_TEST_MS__=1;w.fetch=()=>{externalAttempts++;throw new Error("Unexpected prototype network request");};}});
 const d=dom.window.document;
 const event=(name)=>new dom.window.Event(name,{bubbles:true});
 const click=selector=>{const node=d.querySelector(selector);if(!node)throw new Error("Missing control: "+selector);node.click();};
 const scenario=value=>{const sel=d.querySelector(".scenario-bar select");sel.value=value;sel.dispatchEvent(event("change"));};
 const waitFor=async(predicate)=>{for(let i=0;i<1000;i++){if(predicate())return;await new Promise(r=>setTimeout(r,3));}throw new Error("Timed out waiting for prototype state");};
 click('[data-entry="excel"]');click('[data-action="use"]');
 await waitFor(()=>!!d.querySelector('[data-action="run"]'));
 click('[data-action="run"]');
 assert("Excel reaches an explicitly triggered sample result",d.querySelector(".results")?.textContent.includes("First result"));
 const query=d.querySelector("#query-editor");
 query.value="SELECT 42 AS my_edit;";query.dispatchEvent(event("input"));
 [...d.querySelectorAll("button")].find(b=>b.textContent==="Restart kernel").click();
 assert("Restart disables Run and preserves edited SQL",d.querySelector('[data-action="run"]').disabled&&d.querySelector("#query-editor").value==="SELECT 42 AS my_edit;");
 click('[data-action="restore"]');await waitFor(()=>!!d.querySelector('[data-action="run"]')&&!d.querySelector('[data-action="run"]').disabled);
 assert("Preparation restores readiness without overwriting edited SQL",d.querySelector("#query-editor").value==="SELECT 42 AS my_edit;");
 scenario("failure");click('[data-action="use"]');await waitFor(()=>!!d.querySelector('[data-action="retry"]'));
 click('[data-action="retry"]');await waitFor(()=>!!d.querySelector('[data-action="run"]'));
 assert("Installation retry retains bound Excel input",d.querySelector("#query-editor").value.includes("sales_october.xlsx"));
 scenario("credentials");click('[data-action="use"]');await waitFor(()=>!!d.querySelector('[data-action="credential"]'));
 assert("Provider preparation pauses before credential setup",!d.querySelector(".results")&&!d.querySelector('[data-action="run"]'));
 click('[data-action="credential"]');await waitFor(()=>!!d.querySelector('[data-action="run"]'));
 assert("Credential setup prepares SQL without running inference",!d.querySelector(".results")&&d.querySelector("#query-editor").value.includes("jev_choice(t,"));
 click('[data-action="run"]');
 assert("Explicit provider preview is clearly simulated",d.querySelector(".results")?.textContent.includes("Sample data"));
 scenario("no_recipe");click('[data-action="use"]');await waitFor(()=>!!d.querySelector('[data-action="run"]'));
 assert("Absent recipe still permits generic preparation",d.body.textContent.includes("Ready here")&&d.querySelector('[data-action="run"]').disabled);
 scenario("sql_installed");
 assert("SQL-installed package appears in notebook inventory",d.querySelector('[data-entry="h3"]')?.textContent.includes("Loaded in this runtime"));
 scenario("remote");
 assert("Remote setup requires an explicit local switch",d.querySelector('[data-action="use"]').disabled);
 scenario("incompatible");
 assert("Unavailable build has an actionable explanation",d.querySelector('[data-action="use"]').disabled&&d.body.textContent.includes("No matching build"));
 scenario("happy");
 const filter=[...d.querySelectorAll(".filters button")].find(b=>b.textContent==="Community");filter.click();
 assert("Community entries remain discoverable",d.querySelectorAll("[data-entry]").length===2);
 const input=d.querySelector(".search input");input.value="classify";input.dispatchEvent(event("input"));
 assert("Search understands purpose words",d.querySelectorAll("[data-entry]").length===1&&!!d.querySelector('[data-entry="jev"]'));

 scenario("no_recipe");
 assert("Generic installation includes official documentation",d.querySelector('[data-documentation="inet"]')?.getAttribute("href").includes("duckdb.org/docs/"));
 for(const [label,heading] of [["Full journey","From a task to a result"],["Rules & solver","Automation with explicit execution"],["Product review","What should change in the product"]]){
  [...d.querySelectorAll(".top-tabs button")].find(b=>b.textContent===label).click();
  assert("Design section works: "+label,d.querySelector(".document h2")?.textContent===heading);
 }
 assert("Prototype performs no network or provider requests",externalAttempts===0);
 dom.window.close();off.window.close();
 return {html,tests,evidence,viewSource,css,bytes:new TextEncoder().encode(html).length,track:"B: componentized SSR + inline Preact bundle"};
})();
spur.put("duckdb_journey_components",[{bundle:JSON.stringify({viewSource:duckdbJourneyBuild.viewSource,css:duckdbJourneyBuild.css,evidence:duckdbJourneyBuild.evidence,tests:duckdbJourneyBuild.tests})}]);
({[Symbol.for("Jupyter.display")](){return {"text/html":duckdbJourneyBuild.html};}})


bundle


<!doctype html> DuckDB extension journey | Jute design JUTE · INTERACTIVE DESIGN Extensions, with a purpose Discover a task, prepare DuckDB, get your first result. Prototype · sample data Try the journey Full journey Rules & solver Product review Explore a scenario First-time use Kernel stopped Already installed Installation fails Provider credentials Unavailable build Remote runtime No starter recipe Installed through SQL All setup and results are simulated. Reset Jute / Revenue exploration.ipynb Local DuckDB 1.5.5 Kernel running Restart kernel Extensions Find a tool for your next query. Browse In this notebook Search extensions or tasks All Core Community Excel Core Read Excel worksheets HTTPFS Core Query files from a URL Spatial Core Work with geographic data H3 Community Index coordinates into a grid Jev Community Classify text with AI Full-text search Core Search text in your data INET Core Work with IP addresses JSON Core Read and query JSON 8 sample entries in this prototype. The product uses the full unique Core and Community catalog. START WITH YOUR TASK What would you like to query? Choose a purpose. Jute prepares DuckDB in this notebook and gives you a query to review. Read Excel worksheets Excel · Core Query files from a URL HTTPFS · Core Work with geographic data Spatial · Core Index coordinates into a grid H3 · Community SQL Your prepared query will appear here. Read the complete journey Discover Choose a task Bind inputs Prepare notebook Review query Get results Return & recover Enable active content on this output to explore the interactive paths. Design intent One preparation action handles setup. The user chooses when to run. Search, input values and edited SQL survive recovery. Full unique Core and Community catalog in the product. Eight representative entries here. Enable active content for interactive controls.

In [15]:

var duckdbDiscoveryBuild = await (async () => {
 const shared=JSON.parse(spur.get("duckdb_journey_components").toArray()[0].bundle);
 const {h}=await import("https://esm.sh/preact@10.24.3");
 const {render:ssr}=await import("https://esm.sh/preact-render-to-string@6.5.11");
 const esbuild=await import("npm:esbuild@0.24.0");
 const {JSDOM}=await import("npm:jsdom@24.1.3");
 const addon="\nconst discoveryFacts={\n excel:{fn:\"read_xlsx\",inputs:\"Excel file and worksheet\",output:\"A preview of worksheet rows\",managed:false},\n httpfs:{fn:\"read_parquet\",inputs:\"Remote Parquet URL\",output:\"A bounded preview of remote rows\",managed:true},\n spatial:{fn:\"ST_Point ST_AsText\",inputs:\"Longitude and latitude\",output:\"A geographic point as text\",managed:true},\n h3:{fn:\"h3_latlng_to_cell\",inputs:\"Latitude, longitude and resolution\",output:\"An H3 cell identifier\",managed:false},\n jev:{fn:\"jev_choice\",inputs:\"Source table and text column\",output:\"A team label for each selected request\",managed:true},\n fts:{fn:\"create_fts_index match_bm25\",inputs:\"Custom SQL and source text\",output:\"Results defined by your query\",managed:false},\n inet:{fn:\"INET\",inputs:\"Custom SQL and IP addresses\",output:\"Results defined by your query\",managed:false},\n json:{fn:\"read_json json_extract\",inputs:\"Custom SQL and JSON data\",output:\"Results defined by your query\",managed:false}\n};\nexport const discoveryInitial=()=>({mode:\"compare\",query:\"\",origin:\"all\",selected:\"excel\",journey:null,notice:\"\",setup:false});\nexport function discoveryReduce(s,a){\n if(a.type===\"mode\")return {...s,mode:a.value};\n if(a.type===\"search\")return {...s,query:a.value};\n if(a.type===\"origin\")return {...s,origin:a.value};\n if(a.type===\"clear\")return {...s,query:\"\",origin:\"all\"};\n if(a.type===\"choose\")return {...s,selected:a.id,journey:a.id===s.selected?s.journey:null,notice:\"\",setup:false};\n if(a.type===\"setup\")return {...s,setup:!s.setup};\n if(a.type===\"notice\")return {...s,notice:a.value};\n if(a.type===\"open\")return {...s,journey:s.journey??reduce(initial(),{type:\"select\",id:s.selected}),notice:\"\"};\n if(a.type===\"close\")return {...s,journey:null};\n if(a.type===\"journey\")return {...s,journey:reduce(s.journey,a.action)};\n return s;\n}\nexport function discoveryList(s){\n const terms=s.query.toLowerCase().trim().split(/\\s+/).filter(Boolean);\n return entries.filter(e=>(s.origin===\"all\"||e.source.toLowerCase()===s.origin)&&terms.every(t=>(e.name+\" \"+e.purpose+\" \"+e.tags+\" \"+discoveryFacts[e.id].fn).toLowerCase().includes(t)));\n}\nexport function Discovery({s,act=()=>{},evidence=[]}){\n const e=entries.find(e=>e.id===s.selected),f=discoveryFacts[e.id],list=discoveryList(s);\n const install=\"INSTALL \"+e.id+(e.source===\"Community\"?\" FROM community\":\"\")+\";\\nLOAD \"+e.id+\";\";\n const copy=async()=>{try{await navigator.clipboard.writeText(install);act({type:\"notice\",value:\"Install SQL copied. Run it explicitly in a SQL cell.\"});}catch{act({type:\"notice\",value:\"Select the SQL above to copy it. Nothing was installed.\"});}};\n const current=el(\"section\",{class:\"dc-panel dc-current\",\"data-panel\":\"current\",\"aria-label\":\"Existing UI\"},\n  el(\"p\",{class:\"dc-kicker\"},\"EXISTING UI\"),el(\"h3\",null,f.managed?\"Set up \"+e.name:e.name),\n  el(\"p\",null,f.managed?\"Open the managed setup screen. The primary action follows the current runtime requirements.\":\"Open extension details, then copy installation SQL into a notebook cell.\"),\n  el(\"div\",{class:\"dc-path\"},f.managed?\"Catalog → Set up → Install → Run local check → Activate\":\"Catalog → Extension details → Copy install SQL → SQL cell\"),\n  f.managed?el(\"div\",null,button(s.setup?\"Hide setup steps\":\"View setup steps\",()=>act({type:\"setup\"}),{\"data-action\":\"current-setup\"}),s.setup&&el(\"ol\",{class:\"dc-steps\"},[\"Choose a supported local runtime\",\"Install if needed\",\"Complete required configuration or credentials\",\"Run local check\",\"Enable or activate explicitly\"].map(t=>el(\"li\",null,t))),el(\"p\",{class:\"dc-small\"},\"Function reference and example insertion are available in managed details. Actions here are a reconstruction, not live setup.\")):\n   el(\"div\",null,el(\"pre\",null,el(\"code\",null,install)),button(\"Copy install SQL\",copy,{\"data-action\":\"copy-install\"}),el(\"p\",{class:\"dc-small\"},\"Copying does not install or load the extension.\")),\n  el(\"details\",{class:\"dc-evidence\"},el(\"summary\",null,\"Frontend evidence\"),el(\"p\",null,f.managed?\"DuckDbCatalog.tsx routes reviewed entries to setup. ExtensionSetup.tsx exposes separate readiness actions.\":\"DuckDbCatalog.tsx renders generic details with Copy install SQL.\"),el(\"p\",{class:\"dc-small\"},\"Source reviewed 27 Oct 2026. This panel reconstructs the current interaction; runtime readiness is not observed here.\")));\n const proposed=el(\"section\",{class:\"dc-panel dc-proposed\",\"data-panel\":\"proposed\",\"aria-label\":\"To-be design\"},\n  el(\"p\",{class:\"dc-kicker\"},\"TO-BE DESIGN\"),el(\"h3\",null,e.recipe?e.task:\"Prepare \"+e.name+\" for a custom query\"),\n  el(\"p\",null,e.recipe?\"Start with your task, bind your data, then prepare this notebook in one action.\":\"Keep generic preparation available even when a starter recipe is absent.\"),\n  el(\"dl\",{class:\"dc-facts\"},el(\"dt\",null,\"Inputs\"),el(\"dd\",null,f.inputs),el(\"dt\",null,\"Expected result\"),el(\"dd\",null,f.output),el(\"dt\",null,\"Reference functions\"),el(\"dd\",null,el(\"code\",null,f.fn))),\n  el(\"div\",{class:\"dc-path\"},\"Discover → Choose → Bind inputs → Prepare notebook → Review SQL → Run\"),\n  e.provider&&el(\"p\",{class:\"dc-effect\"},\"Provider-backed task: share only selected text after explicit Run. The preview uses sample data.\"),\n  button(\"Choose this task · Try journey\",()=>act({type:\"open\"}),{class:\"primary\",\"data-action\":\"open-journey\"}),\n  el(\"p\",{class:\"dc-small\"},\"Opens the notebook's existing proposed journey with \"+e.name+\" selected. All preparation and results are simulated.\"));\n return el(\"div\",{class:\"dc-root\",\"data-discovery-root\":true},\n  el(\"header\",{class:\"dc-header\"},el(\"p\",{class:\"dc-kicker\"},\"DUCKDB · DISCOVERY & CHOICE\"),el(\"h1\",null,\"Find the right extension\"),el(\"p\",null,\"Explore what exists today. Choose a task and try the proposed flow.\")),\n  el(\"nav\",{class:\"dc-modes\",\"aria-label\":\"Comparison view\"},[[\"current\",\"Existing UI\"],[\"proposed\",\"To-be design\"],[\"compare\",\"Side-by-side\"]].map(([id,label])=>button(label,()=>act({type:\"mode\",value:id}),{\"aria-pressed\":s.mode===id,\"data-mode\":id}))),\n  el(\"div\",{class:\"dc-layout\"},el(\"aside\",{class:\"dc-browser\",\"aria-label\":\"Discover extensions\"},\n   el(\"label\",{class:\"dc-search\"},el(\"span\",null,\"What do you want to do?\"),el(\"input\",{type:\"search\",placeholder:\"Search a task, extension or function\",value:s.query,\"data-discovery-search\":true,onInput:ev=>act({type:\"search\",value:ev.currentTarget.value})})),\n   el(\"div\",{class:\"dc-origins\",\"aria-label\":\"Extension source\"},[[\"all\",\"All\"],[\"core\",\"Core\"],[\"community\",\"Community\"]].map(([id,label])=>button(label,()=>act({type:\"origin\",value:id}),{\"aria-pressed\":s.origin===id,\"data-origin\":id}))),\n   el(\"p\",{class:\"dc-count\",role:\"status\"},list.length+\" of 8 representative extensions\"),\n   el(\"div\",{class:\"dc-list\"},list.length?list.map(item=>button(el(\"span\",null,el(\"span\",{class:\"dc-card-top\"},el(\"strong\",null,item.name),badge(item.source)),el(\"span\",{class:\"dc-card-purpose\"},item.purpose)),()=>act({type:\"choose\",id:item.id}),{class:\"dc-card\",\"data-choice\":item.id,\"aria-pressed\":s.selected===item.id})):el(\"div\",{class:\"dc-empty\"},el(\"strong\",null,\"No matching extensions\"),el(\"p\",null,\"Try a purpose such as map, spreadsheet or classify.\"),button(\"Clear search and filters\",()=>act({type:\"clear\"}),{\"data-action\":\"clear-discovery\"})))),\n   el(\"main\",{class:\"dc-detail\"},el(\"div\",{class:\"dc-selection\"},el(\"div\",null,el(\"p\",{class:\"dc-kicker\"},\"SELECTED EXTENSION\"),el(\"h2\",{\"data-selected-extension\":e.id},e.name),el(\"p\",null,e.description)),badge(e.source)),\n    el(\"div\",{class:\"dc-comparison \"+(s.mode===\"compare\"?\"dc-both\":\"dc-single\")},s.mode!==\"proposed\"&&current,s.mode!==\"current\"&&proposed),\n    s.notice&&el(\"p\",{class:\"dc-notice\",role:\"status\"},s.notice))),\n  s.journey&&el(\"section\",{class:\"dc-live\",\"aria-label\":\"Selected task journey\",\"data-selected-journey\":s.journey.selected},\n   el(\"div\",{class:\"dc-live-bar\"},el(\"div\",null,el(\"p\",{class:\"dc-kicker\"},\"TRY YOUR CHOICE\"),el(\"h2\",null,entry(s.journey).name+\" · proposed journey\")),button(\"Close journey\",()=>act({type:\"close\"}),{\"data-action\":\"close-journey\"})),\n   h(View,{s:s.journey,act:action=>act({type:\"journey\",action}),evidence})),\n  el(\"footer\",{class:\"dc-footer\"},el(\"span\",null,\"Eight representative extensions. The existing and proposed views are clearly labeled.\"),el(\"span\",null,\"Selections stay in this output session. Re-running the cell resets them.\")));\n}\n";
 const client="\nimport {h,hydrate,render} from \"https://esm.sh/preact@10.24.3\";\nimport {Discovery,discoveryInitial,discoveryReduce} from \"./view.ts\";\nconst root=document.getElementById(\"discovery\");\nlet state=discoveryInitial(),timer;\nconst evidence=JSON.parse(document.getElementById(\"evidence-data\").textContent);\nfunction paint(){render(h(Discovery,{s:state,act,evidence}),root);}\nfunction act(action){\n clearTimeout(timer);\n state=discoveryReduce(state,action);paint();\n if(state.journey?.stage===\"preparing\")timer=setTimeout(()=>act({type:\"journey\",action:{type:\"tick\"}}),window.__JOURNEY_TEST_MS__||480);\n if(action.type===\"open\"){document.querySelector(\".dc-live\")?.scrollIntoView?.({block:\"start\",behavior:\"auto\"});document.querySelector(\".dc-live h2\")?.setAttribute(\"tabindex\",\"-1\");document.querySelector(\".dc-live h2\")?.focus({preventScroll:true});}\n if(action.type===\"close\")document.querySelector('[data-action=\"open-journey\"]')?.focus({preventScroll:true});\n}\nhydrate(h(Discovery,{s:state,act,evidence}),root);\ndocument.documentElement.dataset.interactive=\"true\";\n";
 const css="\n.dc-root{max-width:1280px;margin:auto;padding:28px 22px;font-size:13px}.dc-header{margin-bottom:22px}.dc-kicker{font-size:10px;font-weight:700;letter-spacing:.12em;color:var(--muted);margin:0 0 9px}.dc-header h1{font-size:32px;letter-spacing:-.035em}.dc-header>p:last-child{color:var(--muted);margin-top:12px}.dc-modes{display:flex;gap:6px;flex-wrap:wrap;margin-bottom:20px}.dc-modes button{font-size:12px;padding:8px 12px}.dc-modes button[aria-pressed=true]{background:var(--fg);color:var(--surface);border-color:var(--fg)}.dc-layout{display:grid;grid-template-columns:260px minmax(0,1fr);border:1px solid var(--border);border-radius:10px;overflow:hidden}.dc-browser{background:var(--soft);padding:18px;border-right:1px solid var(--border);min-width:0}.dc-search{display:block;font-size:12px;font-weight:600}.dc-search input{display:block;width:100%;min-width:0;border:1px solid var(--border);border-radius:6px;padding:10px;background:var(--surface);margin-top:9px;font-size:12px}.dc-origins{display:flex;gap:5px;margin:12px 0}.dc-origins button{min-height:34px;padding:5px 9px;font-size:11px}.dc-origins button[aria-pressed=true]{border-color:var(--fg);font-weight:650}.dc-count{font-size:10px;color:var(--muted);margin-bottom:12px}.dc-list{display:flex;flex-direction:column;gap:7px;max-height:550px;overflow:auto;padding:2px}.dc-card{flex-shrink:0;text-align:left;padding:12px;background:var(--surface);min-width:0}.dc-card[aria-pressed=true]{border-color:var(--accent);box-shadow:inset 3px 0 var(--accent)}.dc-card-top{display:flex;align-items:center;justify-content:space-between;gap:8px}.dc-card-top strong{font-size:12px}.dc-card .badge{font-size:9px}.dc-card-purpose{display:block;font-size:11px;color:var(--muted);margin-top:7px;line-height:1.5}.dc-detail{min-width:0;padding:24px}.dc-selection{display:flex;justify-content:space-between;align-items:flex-start;gap:15px;margin-bottom:22px}.dc-selection h2{font-size:25px}.dc-selection p:last-child{font-size:12px;color:var(--muted);margin-top:9px}.dc-comparison{display:grid;gap:16px}.dc-both{grid-template-columns:1fr 1fr}.dc-single{grid-template-columns:1fr}.dc-panel{border:1px solid var(--border);border-radius:8px;padding:20px;min-width:0}.dc-proposed{border-top:3px solid var(--accent)}.dc-panel h3{font-size:17px;line-height:1.3;margin:6px 0 12px}.dc-panel>p,.dc-panel>div>p{font-size:12px;color:var(--muted);line-height:1.65}.dc-panel button{font-size:12px;margin-top:14px;width:100%}.dc-panel pre{white-space:pre-wrap;overflow-wrap:anywhere;background:var(--soft);padding:12px;border-radius:5px;font-size:11px;line-height:1.8;margin:14px 0 0}.dc-path{font-size:11px;line-height:1.9;background:var(--soft);padding:12px;border-radius:5px;margin:16px 0;color:var(--muted)}.dc-facts{display:grid;grid-template-columns:1fr;gap:5px;font-size:12px;margin:15px 0}.dc-facts dt{font-size:10px;font-weight:650;color:var(--muted);margin-top:8px}.dc-facts dd{margin:0;overflow-wrap:anywhere;line-height:1.6}.dc-small{font-size:10px!important;line-height:1.7;margin-top:12px}.dc-evidence{border-top:1px solid var(--border);padding-top:14px;margin-top:18px;font-size:11px}.dc-evidence summary{cursor:pointer;color:var(--muted)}.dc-evidence p{margin-top:9px;line-height:1.7}.dc-steps{font-size:11px;line-height:1.9;padding-left:18px;margin-top:12px}.dc-effect{padding:10px;background:var(--soft);border-left:2px solid var(--accent)}.dc-notice{padding:12px;border:1px solid var(--border);margin-top:16px;font-size:12px}.dc-empty{padding:20px 4px}.dc-empty p{font-size:11px;margin:8px 0 12px;color:var(--muted)}.dc-live{border:1px solid var(--border);border-radius:10px;margin-top:24px;overflow:hidden;scroll-margin-top:20px}.dc-live-bar{padding:20px 24px;background:var(--soft);display:flex;align-items:center;justify-content:space-between;gap:14px;border-bottom:1px solid var(--border)}.dc-live-bar h2{font-size:18px}.dc-live-bar button{font-size:11px}.dc-live .design-header,.dc-live .design-footer{display:none}.dc-live .design-root{padding-top:0}.dc-live .top-tabs{margin-bottom:16px}.dc-footer{display:flex;justify-content:space-between;gap:20px;font-size:10px;line-height:1.7;color:var(--muted);margin-top:18px}.dc-footer span{max-width:48%}@media(max-width:1050px){.dc-both{grid-template-columns:1fr}.dc-layout{grid-template-columns:230px minmax(0,1fr)}.dc-detail{padding:18px}}@media(max-width:700px){.dc-root{padding:20px 12px}.dc-header h1{font-size:27px}.dc-layout{grid-template-columns:1fr}.dc-browser{border-right:0;border-bottom:1px solid var(--border)}.dc-list{max-height:245px}.dc-detail{padding:18px 14px}.dc-panel{padding:16px}.dc-footer{display:block}.dc-footer span{display:block;max-width:none;margin-top:5px}.dc-live-bar{padding:16px}.dc-live-bar h2{font-size:16px}.dc-selection{gap:8px}}\n";
 const dir=await Deno.makeTempDir({prefix:"jute-discovery-choice-"});
 await Deno.writeTextFile(dir+"/view.ts",shared.viewSource+"\n"+addon);
 await Deno.writeTextFile(dir+"/client.ts",client);
 const views=await import("file://"+dir+"/view.ts");
 const baseline=ssr(h(views.Discovery,{s:views.discoveryInitial(),evidence:shared.evidence}));
 const plugin={name:"inline-http",setup(b){
  b.onResolve({filter:/^https?:\/\//},a=>({path:a.path,namespace:"http-url"}));
  b.onResolve({filter:/.*/,namespace:"http-url"},a=>({path:new URL(a.path,a.importer).toString(),namespace:"http-url"}));
  b.onLoad({filter:/.*/,namespace:"http-url"},async a=>{const r=await fetch(a.path);if(!r.ok)throw new Error("Build dependency unavailable: "+r.status);return {contents:await r.text(),loader:"js"};});
 }};
 const built=await esbuild.build({entryPoints:[dir+"/client.ts"],plugins:[plugin],bundle:true,format:"iife",minify:true,write:false,legalComments:"none",target:"es2020"});
 await esbuild.stop();
 const js=built.outputFiles[0].text.replace(/<\/script/gi,"<\\/script");
 const html='<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1"><title>DuckDB Discovery & Choice</title><style>'+shared.css+"\n"+css+'</style></head><body><div id="discovery">'+baseline+'</div><script id="evidence-data" type="application/json">'+JSON.stringify(shared.evidence)+'</script><script>'+js+'</script></body></html>';

 const tests=[];
 const assert=(name,ok)=>{if(!ok)throw new Error("Discovery check failed: "+name);tests.push({name,status:"pass"});};
 const off=new JSDOM(html,{runScripts:"outside-only"});
 assert("Discovery remains readable without scripts",off.window.document.querySelectorAll("[data-choice]").length===8&&off.window.document.querySelectorAll("[data-panel]").length===2);
 let externalAttempts=0;
 const dom=new JSDOM(html,{runScripts:"dangerously",beforeParse(w){w.__JOURNEY_TEST_MS__=1;w.fetch=()=>{externalAttempts++;throw new Error("Unexpected discovery request");};}});
 const d=dom.window.document;
 const event=name=>new dom.window.Event(name,{bubbles:true});
 const click=q=>{const n=d.querySelector(q);if(!n)throw new Error("Missing discovery control: "+q);n.click();};
 const search=value=>{const n=d.querySelector("[data-discovery-search]");n.value=value;n.dispatchEvent(event("input"));};
 const waitFor=async fn=>{for(let i=0;i<1000;i++){if(fn())return;await new Promise(r=>setTimeout(r,3));}throw new Error("Discovery journey timed out");};
 assert("Existing and proposed interactions are labeled",d.querySelector("[data-panel=current]").textContent.includes("Copy install SQL")&&d.querySelector("[data-panel=proposed]").textContent.includes("Choose this task"));
 click("[data-mode=current]");
 assert("Existing UI view shows only the current interaction",!!d.querySelector("[data-panel=current]")&&!d.querySelector("[data-panel=proposed]"));
 click("[data-mode=proposed]");
 assert("To-be view shows only the proposed interaction",!d.querySelector("[data-panel=current]")&&!!d.querySelector("[data-panel=proposed]"));
 click("[data-mode=compare]");
 search("classify");
 assert("Purpose search discovers Jev",d.querySelectorAll("[data-choice]").length===1&&!!d.querySelector('[data-choice="jev"]'));
 search("read_xlsx");
 assert("Function search discovers Excel",d.querySelectorAll("[data-choice]").length===1&&!!d.querySelector('[data-choice="excel"]'));
 search("");click("[data-origin=community]");
 assert("Community filter contains H3 and Jev",d.querySelectorAll("[data-choice]").length===2&&!!d.querySelector('[data-choice="h3"]')&&!!d.querySelector('[data-choice="jev"]'));
 click('[data-choice="h3"]');click('[data-action="open-journey"]');
 assert("Choosing H3 opens the shared H3 task flow",d.querySelector("[data-selected-journey]").getAttribute("data-selected-journey")==="h3"&&d.querySelector(".dc-live .detail-title")?.textContent.includes("H3"));
 click('.dc-live [data-action="use"]');await waitFor(()=>!!d.querySelector('.dc-live [data-action="run"]'));
 assert("Selected task prepares matching SQL without running",d.querySelector(".dc-live #query-editor").value.includes("h3_latlng_to_cell")&&!d.querySelector(".dc-live .results"));
 click('[data-mode="current"]');
 assert("Changing comparison view preserves the prepared journey",d.querySelector(".dc-live #query-editor").value.includes("h3_latlng_to_cell"));
 click('[data-action="close-journey"]');
 assert("Closing the journey preserves source filter and choice",d.querySelector('[data-origin="community"]').getAttribute("aria-pressed")==="true"&&d.querySelector('[data-choice="h3"]').getAttribute("aria-pressed")==="true");
 search("no-such-task");
 assert("Empty search offers recovery",!!d.querySelector('[data-action="clear-discovery"]'));
 click('[data-action="clear-discovery"]');
 assert("Clear restores the full discovery list",d.querySelectorAll("[data-choice]").length===8);
 click('[data-choice="httpfs"]');
 assert("Managed entries show the source-based setup path",d.querySelector("[data-panel=current]").textContent.includes("Run local check")&&!d.querySelector('[data-action="copy-install"]'));
 click('[data-action="current-setup"]');
 assert("Current setup details expand without installation",d.querySelectorAll(".dc-steps li").length===5);
 click('[data-mode="proposed"]');click('[data-choice="inet"]');click('[data-action="open-journey"]');click('.dc-live [data-action="use"]');
 await waitFor(()=>!!d.querySelector('.dc-live [data-action="run"]'));
 assert("A missing recipe retains generic preparation",d.querySelector('.dc-live [data-action="run"]').disabled&&d.querySelector(".dc-live").textContent.includes("Ready here"));
 click('[data-action="close-journey"]');click('[data-choice="jev"]');click('[data-action="open-journey"]');click('.dc-live [data-action="use"]');
 await waitFor(()=>!!d.querySelector('.dc-live [data-action="credential"]'));
 assert("Provider journey pauses at credentials",!d.querySelector('.dc-live [data-action="run"]')&&!d.querySelector(".dc-live .results"));
 assert("Discovery performs no extension or provider requests",externalAttempts===0);
 dom.window.close();off.window.close();

 await Deno.remove(dir,{recursive:true});
 return {html,tests,bytes:new TextEncoder().encode(html).length};
})();
spur.put("duckdb_discovery_checks",duckdbDiscoveryBuild.tests);
({[Symbol.for("Jupyter.display")](){return {"text/html":duckdbDiscoveryBuild.html};}})


name,status
Discovery remains readable without scripts,pass
Existing and proposed interactions are labeled,pass
Existing UI view shows only the current interaction,pass
To-be view shows only the proposed interaction,pass
Purpose search discovers Jev,pass


<!doctype html> DuckDB Discovery & Choice DUCKDB · DISCOVERY & CHOICE Find the right extension Explore what exists today. Choose a task and try the proposed flow. Existing UI To-be design Side-by-side What do you want to do? All Core Community 8 of 8 representative extensions Excel Core Read Excel worksheets HTTPFS Core Query files from a URL Spatial Core Work with geographic data H3 Community Index coordinates into a grid Jev Community Classify text with AI Full-text search Core Search text in your data INET Core Work with IP addresses JSON Core Read and query JSON SELECTED EXTENSION Excel Bring an .xlsx worksheet into a SQL query. Core EXISTING UI Excel Open extension details, then copy installation SQL into a notebook cell. Catalog → Extension details → Copy install SQL → SQL cell INSTALL excel;
LOAD excel; Copy install SQL Copying does not install or load the extension. Frontend evidence DuckDbCatalog.tsx renders generic details with Copy install SQL. Source reviewed 27 Oct 2026. This panel reconstructs the current interaction; runtime readiness is not observed here. TO-BE DESIGN Read a worksheet Start with your task, bind your data, then prepare this notebook in one action. Inputs Excel file and worksheet Expected result A preview of worksheet rows Reference functions read_xlsx Discover → Choose → Bind inputs → Prepare notebook → Review SQL → Run Choose this task · Try journey Opens the notebook's existing proposed journey with Excel selected. All preparation and results are simulated. Eight representative extensions. The existing and proposed views are clearly labeled. Selections stay in this output session. Re-running the cell resets them.

# Journey map and task recipes

The main path is **discover → choose a purpose → bind inputs → prepare this notebook → review SQL → run → use the result**.

| Stage | User intent | Interface behavior | Recovery and continuity |
|---|---|---|---|
| Discover | Find something for my task | Search by purpose, package or function; Core and Community filters; full unique catalog | Preserve search and filters on Back. Refresh failures retain cached entries. |
| Choose | Understand what this enables | Purpose first, expected output, starter tasks, source and documentation | A missing recipe still permits generic installation. |
| Bind | Use my data | File picker, URL, or table and column selection; inline validation | Keep values when setup fails. Never require manual SQL substitution for a provided recipe. |
| Prepare | Make this work in this notebook | One action starts/reuses the kernel, checks compatibility, reuses cache, installs, loads and verifies | Show progress. Retry resumes with the same task. Request credentials only when needed. |
| Review | See what will run | Prepare an editable SQL cell; show target and relevant data effects | Preserve user edits. Preparation does not execute a task query. |
| Run | Get a first useful result | Explicit Run preview, bounded result display, follow-up querying | Keep query and inputs on query errors or empty results. |
| Return | Continue later or share | Read actual runtime inventory, including SQL-installed extensions; record notebook dependency intent | Recheck readiness after restart or runtime change. Keep credentials out of notebook files. |

## Representative purposes

| Extension | Predefined purpose | Input binding | Query/result | Execution effect |
|---|---|---|---|---|
| Excel (Core) | Read a worksheet | .xlsx file, sheet, header option | `read_xlsx` preview | Local file read |
| HTTPFS (Core) | Preview remote Parquet | URL | `read_parquet` preview | Network request when run |
| Spatial (Core) | Create a geographic point | Longitude and latitude | `ST_AsText(ST_Point(...))` | Local computation |
| H3 (Community) | Find a grid cell | Latitude, longitude, resolution | `h3_latlng_to_cell` | Local computation |
| Jev (Community) | Route support requests | Source table and selected text column | `jev_choice(row, question, options)` | Provider-backed inference when run; API charges may apply |
| Entry without a recipe | Prepare for a custom query | Package/source, current runtime | Ready state, documentation, editable SQL cell | Determined by the user’s query |

The prototype contains eight representative catalog entries. It demonstrates the shared interaction pattern, rather than reproducing the entire extension registry. All results are labeled sample data. The H3 result is illustrative; custom SQL is retained but not executed.

## State and lifecycle rules

- **Core / Community:** package origin, independent of recipe availability.
- **Installed:** a compatible package is available in the installation cache.
- **Ready here:** loaded and verified for the current notebook runtime.
- **After restart:** preserve cached installation and SQL; invalidate old readiness and prepare again before Run.
- **Already installed:** reuse the compatible cache instead of forcing another download.
- **Installed through SQL:** reconcile observed inventory with the catalog and show it in the notebook list.
- **No available build:** explain the selected version/platform problem and offer another runtime or package.
- **Remote/custom runtime:** retain inspection and manual instructions; managed setup requires an explicit compatible target selection.
- **Remove automatic preparation:** stop future restoration for this notebook; explain that loaded code can remain until kernel restart.

## Notebook dependency proposal

Save the source-qualified package identity and preparation intent with the notebook. Reopening resolves those dependencies for the current DuckDB version and platform. Never persist an API token or treat a previous runtime’s verification as current readiness.

## Source references

The source-based evaluation from this conversation identified generic details ending at copied installation SQL, the guided setup stages, limited portable examples, and managed-only inventory. Relevant files are `DuckDbCatalog.tsx`, `ExtensionSetup.tsx`, `model.ts`, `ExtensionCapabilities.tsx`, `exampleInsertion.ts`, and `useNotebookExtensionData.ts` in `jute-notebook/src/ui/`.

Official task references: [Excel in DuckDB 1.5](https://duckdb.org/docs/1.5/core_extensions/excel), [H3 Community extension](https://duckdb.org/community_extensions/extensions/h3), [Jev Community extension](https://duckdb.org/community_extensions/extensions/jev). Jev’s exact `jev_choice(row, question, options)` signature and row-sharing semantics were also read through `notebook_duckdb_functions` from the bundled versioned reference. That reference is not live readiness evidence.


# Bounded workflow evidence

The solver evaluated declared finite states and transitions using the implemented `workflow` catalog. It checks the proposed workflow, not the running application or its usability. Trace bounds are the supplied event counts, not click counts or timing claims.

| Modeled scenario | Expected | Solver | Receipt |
|---|---|---|---|
| First useful result | Allow | sat / pass | `sol_fecf6b91ad9e4a83` |
| Retry after installation failure | Allow | sat / pass | `sol_8ada51e285e44a8f` |
| Credential-based explicit use | Allow | sat / pass | `sol_64c34fda24744ea0` |
| Skip runtime verification | Reject | unsat / fail | `sol_64f4fcbe1f9b4ef6` |
| Use stale readiness after restart | Reject | unsat / fail | `sol_bf51a29bfe304dbe` |
| Send provider request during preparation | Reject | unsat / fail | `sol_e83d287763c64063` |

Each request binds `workflow.initial_state_allowed`, `workflow.transition_allowed`, and `workflow.bounded_reachability` with the trace length as the bound. The rejected cases fail the allowed-transition rule. A final state labeled Result does not make an invalid path valid.

The model assumes caller-resolved guards, successful installation when the success event is supplied, and load/verification for the selected runtime. It does not prove actual downloads, credential validity, correct SQL results, input preservation or production readiness. Those require implementation and interaction checks.

<details><summary>Exact normalized model and concrete traces</summary>

```json
{
  "schema": "workflow bounded trace",
  "initial_states": [
    "Browse"
  ],
  "target_states": [
    "Result"
  ],
  "edges": [
    [
      "Browse",
      "choose_task",
      "Inputs"
    ],
    [
      "Inputs",
      "use_in_notebook",
      "PreparationRequested"
    ],
    [
      "PreparationRequested",
      "start_or_reuse_runtime",
      "RuntimeReady"
    ],
    [
      "RuntimeReady",
      "install_or_reuse_cache",
      "Installed"
    ],
    [
      "RuntimeReady",
      "installation_failed",
      "InstallFailed"
    ],
    [
      "InstallFailed",
      "retry",
      "RuntimeReady"
    ],
    [
      "Installed",
      "load_and_verify_current_runtime",
      "Ready"
    ],
    [
      "Installed",
      "credential_required",
      "NeedsCredential"
    ],
    [
      "NeedsCredential",
      "save_demo_credential_and_verify",
      "Ready"
    ],
    [
      "Ready",
      "prepare_editable_query",
      "QueryPrepared"
    ],
    [
      "QueryPrepared",
      "explicit_run",
      "Result"
    ],
    [
      "QueryPrepared",
      "restart_runtime",
      "NeedsPreparation"
    ],
    [
      "NeedsPreparation",
      "start_or_reuse_runtime",
      "RuntimeReady"
    ]
  ],
  "traces": [
    {
      "id": "local_task",
      "states": [
        "Browse",
        "Inputs",
        "PreparationRequested",
        "RuntimeReady",
        "Installed",
        "Ready",
        "QueryPrepared",
        "Result"
      ],
      "events": [
        "choose_task",
        "use_in_notebook",
        "start_or_reuse_runtime",
        "install_or_reuse_cache",
        "load_and_verify_current_runtime",
        "prepare_editable_query",
        "explicit_run"
      ],
      "expected": "pass",
      "horizon": 7
    },
    {
      "id": "retry",
      "states": [
        "Browse",
        "Inputs",
        "PreparationRequested",
        "RuntimeReady",
        "InstallFailed",
        "RuntimeReady",
        "Installed",
        "Ready",
        "QueryPrepared",
        "Result"
      ],
      "events": [
        "choose_task",
        "use_in_notebook",
        "start_or_reuse_runtime",
        "installation_failed",
        "retry",
        "install_or_reuse_cache",
        "load_and_verify_current_runtime",
        "prepare_editable_query",
        "explicit_run"
      ],
      "expected": "pass",
      "horizon": 9
    },
    {
      "id": "provider",
      "states": [
        "Browse",
        "Inputs",
        "PreparationRequested",
        "RuntimeReady",
        "Installed",
        "NeedsCredential",
        "Ready",
        "QueryPrepared",
        "Result"
      ],
      "events": [
        "choose_task",
        "use_in_notebook",
        "start_or_reuse_runtime",
        "install_or_reuse_cache",
        "credential_required",
        "save_demo_credential_and_verify",
        "prepare_editable_query",
        "explicit_run"
      ],
      "expected": "pass",
      "horizon": 8
    },
    {
      "id": "skip_check",
      "states": [
        "Browse",
        "Inputs",
        "PreparationRequested",
        "RuntimeReady",
        "Installed",
        "QueryPrepared",
        "Result"
      ],
      "events": [
        "choose_task",
        "use_in_notebook",
        "start_or_reuse_runtime",
        "install_or_reuse_cache",
        "prepare_editable_query",
        "explicit_run"
      ],
      "expected": "fail",
      "horizon": 6
    },
    {
      "id": "stale_runtime",
      "states": [
        "Browse",
        "Inputs",
        "PreparationRequested",
        "RuntimeReady",
        "Installed",
        "Ready",
        "QueryPrepared",
        "NeedsPreparation",
        "Result"
      ],
      "events": [
        "choose_task",
        "use_in_notebook",
        "start_or_reuse_runtime",
        "install_or_reuse_cache",
        "load_and_verify_current_runtime",
        "prepare_editable_query",
        "restart_runtime",
        "explicit_run"
      ],
      "expected": "fail",
      "horizon": 8
    },
    {
      "id": "auto_provider",
      "states": [
        "Browse",
        "Inputs",
        "PreparationRequested",
        "RuntimeReady",
        "Installed",
        "NeedsCredential",
        "Ready",
        "QueryPrepared",
        "Result"
      ],
      "events": [
        "choose_task",
        "use_in_notebook",
        "start_or_reuse_runtime",
        "install_or_reuse_cache",
        "credential_required",
        "save_demo_credential_and_verify",
        "prepare_editable_query",
        "automatic_provider_request"
      ],
      "expected": "fail",
      "horizon": 8
    }
  ]
}
```

To reproduce each request, repeat the edge relation at every event index, use the trace's states/events as facts, set safe_states to the full declared state domain (no safety_invariant rule is bound), and bind the three catalog rules above to that trace. Exact persisted solver envelopes can be reloaded by receipt through `get_solve_result`.

</details>


In [16]:
({
 [Symbol.for("Jupyter.display")]() {
  const journey=JSON.parse(spur.get("duckdb_journey_components").toArray()[0].bundle).tests;
  const discovery=spur.get("duckdb_discovery_checks").toArray();
  const all=[...journey,...discovery];
  return {"text/markdown":"## Artifact verification\n\n"+
   all.length+" automated document and DOM checks passed: "+journey.length+" journey checks and "+discovery.length+" discovery/comparison checks. Both interactive outputs have a readable scripts-off baseline and inline runtime assets.\n\n"+
   "**Native webview walkthrough · 27 October 2026:** 15 checks passed for comparison modes, purpose/function search, Community filtering, H3 selection, focus transfer, matching SQL, explicit sample results, preserved selection and empty-search recovery. Responsive checks passed at 375, 652 and 1200 CSS pixels, including complete card text and no horizontal overflow.\n\n"+
   "**Try it:** find the Discovery & Choice output, compare Existing UI with To-be design, choose an extension, then select Choose this task · Try journey. Close the journey to keep browsing with the same search, filter and selection. Re-running resets the output session.\n\n"+
   "**Reactive wiring:** original journey builder → duckdb_journey_components → discovery cell → duckdb_discovery_checks → this report. Re-running the builder cascades both downstream cells.\n\n"+
   "**Scope:** the Existing UI panel reconstructs the reviewed frontend source. Setup, credentials, SQL execution and query results remain simulated. Clipboard behavior depends on host permissions; no installation or provider request occurs. These checks do not establish production DuckDB readiness or a full accessibility audit.\n\n"+
   "<details><summary>Automated check details</summary>\n\n"+
   all.map(t=>"- "+t.name+": "+t.status).join("\n")+"\n\n</details>"};
 }
})

## Artifact verification

39 automated document and DOM checks passed: 21 journey checks and 18 discovery/comparison checks. Both interactive outputs have a readable scripts-off baseline and inline runtime assets.

**Native webview walkthrough · 27 October 2026:** 15 checks passed for comparison modes, purpose/function search, Community filtering, H3 selection, focus transfer, matching SQL, explicit sample results, preserved selection and empty-search recovery. Responsive checks passed at 375, 652 and 1200 CSS pixels, including complete card text and no horizontal overflow.

**Try it:** find the Discovery & Choice output, compare Existing UI with To-be design, choose an extension, then select Choose this task · Try journey. Close the journey to keep browsing with the same search, filter and selection. Re-running resets the output session.

**Reactive wiring:** original journey builder → duckdb_journey_components → discovery cell → duckdb_discovery_checks → this report. Re-running the builder cascades both downstream cells.

**Scope:** the Existing UI panel reconstructs the reviewed frontend source. Setup, credentials, SQL execution and query results remain simulated. Clipboard behavior depends on host permissions; no installation or provider request occurs. These checks do not establish production DuckDB readiness or a full accessibility audit.

<details><summary>Automated check details</summary>

- One self-contained document: pass
- No banned separator glyphs: pass
- Meaningful scripts-off baseline: pass
- Excel reaches an explicitly triggered sample result: pass
- Restart disables Run and preserves edited SQL: pass
- Preparation restores readiness without overwriting edited SQL: pass
- Installation retry retains bound Excel input: pass
- Provider preparation pauses before credential setup: pass
- Credential setup prepares SQL without running inference: pass
- Explicit provider preview is clearly simulated: pass
- Absent recipe still permits generic preparation: pass
- SQL-installed package appears in notebook inventory: pass
- Remote setup requires an explicit local switch: pass
- Unavailable build has an actionable explanation: pass
- Community entries remain discoverable: pass
- Search understands purpose words: pass
- Generic installation includes official documentation: pass
- Design section works: Full journey: pass
- Design section works: Rules & solver: pass
- Design section works: Product review: pass
- Prototype performs no network or provider requests: pass
- Discovery remains readable without scripts: pass
- Existing and proposed interactions are labeled: pass
- Existing UI view shows only the current interaction: pass
- To-be view shows only the proposed interaction: pass
- Purpose search discovers Jev: pass
- Function search discovers Excel: pass
- Community filter contains H3 and Jev: pass
- Choosing H3 opens the shared H3 task flow: pass
- Selected task prepares matching SQL without running: pass
- Changing comparison view preserves the prepared journey: pass
- Closing the journey preserves source filter and choice: pass
- Empty search offers recovery: pass
- Clear restores the full discovery list: pass
- Managed entries show the source-based setup path: pass
- Current setup details expand without installation: pass
- A missing recipe retains generic preparation: pass
- Provider journey pauses at credentials: pass
- Discovery performs no extension or provider requests: pass

</details>